# 10 · EMC-BUILT 2022 — reproducible non-urban reference and derived SUHI diagnostics

**Study:** Retrospective LST modeling and derived SUHI diagnostics in Barranquilla  
**Scope:** Retrospective component, 2013–2025  
**Purpose:** Reproducibly construct the non-urban reference \(R\), audit full-domain M5 outputs, define the urban support \(U\), derive SUHI for 2023–2025, perform sensitivity and error-decomposition analyses, export the final products, and generate the 2025 SUHI figure.

### Frozen methodological definitions
- LST is the only directly modeled variable.
- SUHI is a derived diagnostic.
- Independent TEST period: 2023–2025.
- \(BU_{all}\): built-up fraction ≥ 10%.
- \(BU_{core}\): connected components ≥ 0.333 km².
- Reference \(R\): 4–6 km from \(BU_{core}\), within the fixed TEST support and excluding \(BU_{all}\).
- Final urban support: \(U=BU_{core}\cap M_{common,Y}\).
- Final P95: upper tail of SUHI within the urban support.
- SUHI is retained as a continuous variable; qualitative intensity classes are not used.

> This notebook supersedes the pilot/exploratory workflow and contains the finalized reproducible analytical chain.


## 00. Entorno y montaje


In [ ]:

# ============================================================
# 00. Google Drive — montaje robusto para Google Colab
# ============================================================

from pathlib import Path
import os
import shutil

from google.colab import drive

MOUNT_POINT = Path("/content/drive")
PROJECT_DIR = (
    MOUNT_POINT
    / "MyDrive"
    / "Barranquilla_LST_2013_2026_Modeling"
)

def is_active_mount(mount_point="/content/drive"):
    """Comprueba si el punto aparece como filesystem montado."""
    with open("/proc/mounts", "r", encoding="utf-8") as f:
        mounts = f.read().splitlines()
    return any(
        line.split()[1] == mount_point
        for line in mounts
        if len(line.split()) >= 2
    )

if not is_active_mount():

    # Colab puede dejar una carpeta residual /content/drive
    # después de reiniciar el runtime.
    if MOUNT_POINT.exists() and any(MOUNT_POINT.iterdir()):
        stale = Path("/content/drive_stale")

        if stale.exists():
            shutil.rmtree(stale)

        MOUNT_POINT.rename(stale)
        print(f"Carpeta residual movida a: {stale}")

    drive.mount(str(MOUNT_POINT))

print("Drive montado:", is_active_mount())
print("Proyecto disponible:", PROJECT_DIR.exists())

if not PROJECT_DIR.exists():
    raise FileNotFoundError(
        f"No se encontró el proyecto en:\n{PROJECT_DIR}"
    )


## 01. Configuración de rutas

Las rutas siguen la estructura actual del proyecto. El TIFF EMC-BUILT original se mantiene intacto en `raw/`.

In [ ]:
from pathlib import Path

ROOT = Path('/content/drive/MyDrive/Barranquilla_LST_2013_2026_Modeling')

EMC_PATH = ROOT / '01_aoi_reference' / 'emc_built_2022' / 'raw' / \
    'EMC_BUILT_S_E2022_GLOBE_R2025A_54009_10_V1_0_R8_C11.tif'

REF_PATH = ROOT / '01_aoi_reference' / 'reference_grid' / 'model_domain_reference.tif'

PROCESSED_DIR = ROOT / '01_aoi_reference' / 'emc_built_2022' / 'processed'
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print('EMC-BUILT:', EMC_PATH)
print('Referencia:', REF_PATH)
print('Salida:', PROCESSED_DIR)

assert EMC_PATH.exists(), f'No se encontró EMC-BUILT: {EMC_PATH}'
assert REF_PATH.exists(), f'No se encontró la grilla de referencia: {REF_PATH}'

print('\n✓ Rutas verificadas correctamente.')


## 02. Inspección técnica de los dos rásteres

Se revisan las propiedades geométricas y radiométricas antes de decidir el procedimiento de reproyección.

In [ ]:
import rasterio

def inspect_raster(name, path):
    with rasterio.open(path) as src:
        print('=' * 80)
        print(name)
        print('=' * 80)
        print('Archivo     :', path.name)
        print('CRS         :', src.crs)
        print('Shape       :', (src.height, src.width))
        print('Resolución  :', src.res)
        print('Bounds      :', src.bounds)
        print('Transform   :', src.transform)
        print('Dtype       :', src.dtypes[0])
        print('NoData      :', src.nodata)
        print('Band count  :', src.count)
        print()

inspect_raster('EMC-BUILT 2022 ORIGINAL', EMC_PATH)
inspect_raster('GRILLA CANÓNICA DEL PROYECTO', REF_PATH)


## 03. Comprobaciones esperadas

Antes de continuar, confirmar manualmente que:

- EMC-BUILT está en **World Mollweide (ESRI:54009)**.
- EMC-BUILT tiene resolución nominal de **10 m**.
- EMC-BUILT utiliza **NoData = 255**.
- La grilla canónica está en **EPSG:32618**.
- La resolución de la grilla canónica es **30 m**.

La siguiente etapa será la reproyección **conservativa de superficie** y la alineación exacta a la grilla canónica. No ejecutar una reproyección convencional con bilinear o nearest-neighbor para esta variable extensiva.

In [ ]:
# Resumen automático de las comprobaciones básicas
with rasterio.open(EMC_PATH) as emc, rasterio.open(REF_PATH) as ref:
    checks = {
        'EMC resolución ≈ 10 m': abs(emc.res[0] - 10) < 1e-6 and abs(abs(emc.res[1]) - 10) < 1e-6,
        'EMC NoData = 255': emc.nodata == 255,
        'Referencia EPSG:32618': ref.crs is not None and ref.crs.to_epsg() == 32618,
        'Referencia resolución = 30 m': abs(ref.res[0] - 30) < 1e-6 and abs(abs(ref.res[1]) - 30) < 1e-6,
    }

for k, v in checks.items():
    print(('✓' if v else '✗'), k)

print('\nNo continuar con la reproyección hasta revisar esta salida.')


## 04. Alineación conservativa de EMC-BUILT y fracción built-up


In [ ]:
from rasterio.warp import transform_bounds

with rasterio.open(EMC_PATH) as emc, rasterio.open(REF_PATH) as ref:

    ref_bounds_in_emc = transform_bounds(
        ref.crs,
        emc.crs,
        *ref.bounds,
        densify_pts=21
    )

    print("Dominio Landsat expresado en CRS EMC-BUILT:")
    print(ref_bounds_in_emc)

    print("\nBounds de la tesela EMC-BUILT:")
    print(emc.bounds)

    inside = (
        ref_bounds_in_emc[0] >= emc.bounds.left
        and ref_bounds_in_emc[1] >= emc.bounds.bottom
        and ref_bounds_in_emc[2] <= emc.bounds.right
        and ref_bounds_in_emc[3] <= emc.bounds.top
    )

    print("\nDominio completamente contenido en la tesela EMC:", inside)

In [ ]:
import numpy as np
import rasterio
from rasterio.warp import reproject, Resampling

OUT_ALIGNED = (
    PROCESSED_DIR /
    "EMC_BUILT_S_E2022_EPSG32618_30m_aligned.tif"
)

DST_NODATA = -9999.0

with rasterio.open(EMC_PATH) as src, rasterio.open(REF_PATH) as ref:

    # Matriz destino: exactamente la geometría canónica Landsat
    dst = np.full(
        (ref.height, ref.width),
        DST_NODATA,
        dtype=np.float32
    )

    reproject(
        source=rasterio.band(src, 1),
        destination=dst,

        src_transform=src.transform,
        src_crs=src.crs,
        src_nodata=src.nodata,

        dst_transform=ref.transform,
        dst_crs=ref.crs,
        dst_nodata=DST_NODATA,

        resampling=Resampling.sum,
        init_dest_nodata=True
    )

    profile = ref.profile.copy()

    profile.update(
        dtype="float32",
        count=1,
        nodata=DST_NODATA,
        compress="deflate",
        predictor=3
    )

    with rasterio.open(OUT_ALIGNED, "w", **profile) as dst_file:
        dst_file.write(dst, 1)

print("✓ Archivo generado:")
print(OUT_ALIGNED)

In [ ]:
with rasterio.open(OUT_ALIGNED) as emc30, rasterio.open(REF_PATH) as ref:

    print("CRS:")
    print(" EMC :", emc30.crs)
    print(" REF :", ref.crs)

    print("\nShape:")
    print(" EMC :", emc30.shape)
    print(" REF :", ref.shape)

    print("\nResolución:")
    print(" EMC :", emc30.res)
    print(" REF :", ref.res)

    print("\nBounds:")
    print(" EMC :", emc30.bounds)
    print(" REF :", ref.bounds)

    print("\nTransform iguales:")
    print(emc30.transform == ref.transform)

    data = emc30.read(1)

    valid = (
        np.isfinite(data) &
        (data != emc30.nodata)
    )

    vals = data[valid]

    print("\n--- EMC-BUILT reproyectado ---")
    print("Píxeles válidos :", vals.size)
    print("Mínimo          :", vals.min())
    print("Máximo          :", vals.max())
    print("Media           :", vals.mean())
    print("Suma built-up   :", vals.sum(), "m²")

In [ ]:
# 07. Conversión a fracción built-up y diagnóstico básico

with rasterio.open(OUT_ALIGNED) as src:
    built_m2 = src.read(1).astype("float32")
    nodata = src.nodata
    profile = src.profile.copy()

valid = np.isfinite(built_m2) & (built_m2 != nodata)

built_fraction = np.full_like(built_m2, np.nan, dtype=np.float32)
built_fraction[valid] = built_m2[valid] / 900.0

vals = built_fraction[valid]

print("--- Fracción built-up ---")
print("Mínimo :", np.nanmin(vals))
print("Máximo :", np.nanmax(vals))
print("Media  :", np.nanmean(vals))

for p in [50, 75, 90, 95, 99, 99.5]:
    print(f"P{p:<4}: {np.nanpercentile(vals, p):.4f}")

for threshold in [0.10, 0.25, 0.50]:
    n = np.sum(vals >= threshold)
    area_cells_km2 = n * 900 / 1e6

    print(
        f"\n>= {threshold:.0%}: "
        f"{n:,} píxeles | "
        f"{area_cells_km2:.2f} km² de celdas"
    )

In [ ]:
OUT_FRACTION = (
    PROCESSED_DIR /
    "EMC_BUILT_fraction_E2022_EPSG32618_30m.tif"
)

fraction_export = np.full(
    built_fraction.shape,
    -9999.0,
    dtype=np.float32
)

fraction_export[valid] = built_fraction[valid]

profile.update(
    dtype="float32",
    nodata=-9999.0,
    compress="deflate",
    predictor=3
)

with rasterio.open(OUT_FRACTION, "w", **profile) as dst:
    dst.write(fraction_export, 1)

print("✓ Guardado:")
print(OUT_FRACTION)

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap

fig, axes = plt.subplots(2, 2, figsize=(14, 12))

# Fracción continua
im = axes[0, 0].imshow(
    np.where(valid, built_fraction, np.nan),
    vmin=0,
    vmax=1
)

axes[0, 0].set_title("EMC-BUILT fraction 2022")
axes[0, 0].axis("off")
plt.colorbar(im, ax=axes[0, 0], fraction=0.046)

# Umbrales
for ax, threshold in zip(
    [axes[0, 1], axes[1, 0], axes[1, 1]],
    [0.10, 0.25, 0.50]
):
    mask = valid & (built_fraction >= threshold)

    ax.imshow(mask, interpolation="nearest")
    ax.set_title(f"Built-up fraction ≥ {threshold:.0%}")
    ax.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import pandas as pd

from scipy import ndimage

PIXEL_AREA_M2 = 30 * 30
PIXEL_AREA_KM2 = PIXEL_AREA_M2 / 1e6

def connected_component_stats(mask, threshold_label):
    # Estructura 3x3 = conectividad de 8 vecinos
    structure = np.ones((3, 3), dtype=np.uint8)

    labels, n_components = ndimage.label(
        mask.astype(np.uint8),
        structure=structure
    )

    component_ids, counts = np.unique(
        labels[labels > 0],
        return_counts=True
    )

    df = pd.DataFrame({
        "component_id": component_ids,
        "n_pixels": counts
    })

    df["area_km2"] = df["n_pixels"] * PIXEL_AREA_KM2

    df = df.sort_values(
        "area_km2",
        ascending=False
    ).reset_index(drop=True)

    df["rank"] = np.arange(1, len(df) + 1)

    total_area = df["area_km2"].sum()

    df["area_fraction"] = df["area_km2"] / total_area
    df["cumulative_fraction"] = df["area_fraction"].cumsum()

    print("=" * 70)
    print(f"UMBRAL {threshold_label}")
    print("=" * 70)
    print(f"Número de componentes : {n_components:,}")
    print(f"Área total máscara     : {total_area:.2f} km²")

    if len(df) > 0:
        print(f"Componente principal   : {df.iloc[0]['area_km2']:.2f} km²")
        print(
            f"% del área en componente principal: "
            f"{100 * df.iloc[0]['area_fraction']:.2f}%"
        )

    print("\nTop 20 componentes:")
    display(
        df[
            [
                "rank",
                "component_id",
                "n_pixels",
                "area_km2",
                "area_fraction",
                "cumulative_fraction"
            ]
        ].head(20)
    )

    return labels, df

In [ ]:
mask_10 = valid & (built_fraction >= 0.10)
mask_25 = valid & (built_fraction >= 0.25)

labels_10, stats_10 = connected_component_stats(
    mask_10,
    "10 %"
)

labels_25, stats_25 = connected_component_stats(
    mask_25,
    "25 %"
)

## 05. Componentes built-up, análisis rank–size y definición de `BU_core`


In [ ]:
import rasterio
from rasterio.transform import xy
from pyproj import Transformer

transformer = Transformer.from_crs(
    "EPSG:32618",
    "EPSG:4326",
    always_xy=True
)

rows = []

with rasterio.open(OUT_ALIGNED) as src:
    tr = src.transform

    for _, r in stats_10.head(20).iterrows():

        cid = int(r["component_id"])

        rr, cc = np.where(labels_10 == cid)

        row_c = int(np.mean(rr))
        col_c = int(np.mean(cc))

        x, y = xy(tr, row_c, col_c)

        lon, lat = transformer.transform(x, y)

        rows.append({
            "rank": int(r["rank"]),
            "component_id": cid,
            "area_km2": r["area_km2"],
            "x_utm": x,
            "y_utm": y,
            "longitude": lon,
            "latitude": lat
        })

components_location = pd.DataFrame(rows)

display(components_location)

In [ ]:
# 14. Inventario de límites disponibles

BOUNDARY_DIR = ROOT / "01_aoi_reference" / "boundary"

print("Contenido de:")
print(BOUNDARY_DIR)
print()

for p in sorted(BOUNDARY_DIR.rglob("*")):
    if p.is_file():
        print(p.relative_to(BOUNDARY_DIR))

In [ ]:
# 14. Curva rank–size — componentes built-up >= 10 %

rank_area = stats_10[["rank", "component_id", "area_km2"]].copy()

fig, ax = plt.subplots(figsize=(9, 6))

ax.plot(
    rank_area["rank"],
    rank_area["area_km2"],
    marker="o",
    markersize=3,
    linewidth=1
)

ax.set_xlabel("Rango del componente")
ax.set_ylabel("Área del componente (km²)")
ax.set_title("Rank–size de componentes built-up ≥ 10 %")
ax.set_xlim(1, min(100, len(rank_area)))

plt.show()

In [ ]:
# 15. Curva rank–size en escala log-log

top_n = min(100, len(rank_area))

fig, ax = plt.subplots(figsize=(9, 6))

ax.plot(
    np.log10(rank_area["rank"].iloc[:top_n]),
    np.log10(rank_area["area_km2"].iloc[:top_n]),
    marker="o",
    markersize=4,
    linewidth=1
)

ax.set_xlabel("log10(rango)")
ax.set_ylabel("log10(área km²)")
ax.set_title("Rank–size log-log — top 100 componentes")

plt.show()

In [ ]:
# 16. Saltos relativos entre componentes consecutivos

rank_area["next_area_km2"] = rank_area["area_km2"].shift(-1)

rank_area["area_ratio"] = (
    rank_area["area_km2"] /
    rank_area["next_area_km2"]
)

rank_area["relative_drop"] = (
    rank_area["area_km2"] -
    rank_area["next_area_km2"]
) / rank_area["area_km2"]

display(
    rank_area[
        [
            "rank",
            "component_id",
            "area_km2",
            "next_area_km2",
            "area_ratio",
            "relative_drop"
        ]
    ].head(30)
)

In [ ]:
# 17. Razón de área entre componentes consecutivos

top_n = min(50, len(rank_area) - 1)

fig, ax = plt.subplots(figsize=(10, 5))

ax.plot(
    rank_area["rank"].iloc[:top_n],
    rank_area["area_ratio"].iloc[:top_n],
    marker="o"
)

ax.set_xlabel("Rango del componente")
ax.set_ylabel("Área componente i / área componente i+1")
ax.set_title("Saltos de tamaño entre componentes consecutivos")

plt.show()

In [ ]:
# 18. Breakpoint mediante regresión segmentada log-log
# Se excluye el componente principal (rank 1)

from sklearn.linear_model import LinearRegression

subset = rank_area.iloc[1:50].copy()

x = np.log10(subset["rank"].values).reshape(-1, 1)
y = np.log10(subset["area_km2"].values)

results = []

# mínimo 4 observaciones por segmento
for split in range(4, len(subset) - 4):

    x1, y1 = x[:split], y[:split]
    x2, y2 = x[split:], y[split:]

    m1 = LinearRegression().fit(x1, y1)
    m2 = LinearRegression().fit(x2, y2)

    pred1 = m1.predict(x1)
    pred2 = m2.predict(x2)

    sse1 = np.sum((y1 - pred1) ** 2)
    sse2 = np.sum((y2 - pred2) ** 2)

    results.append({
        "split_index": split,
        "break_rank": int(subset.iloc[split]["rank"]),
        "break_area_km2": float(subset.iloc[split]["area_km2"]),
        "sse_total": float(sse1 + sse2)
    })

break_df = pd.DataFrame(results)

best = break_df.loc[
    break_df["sse_total"].idxmin()
]

print("Breakpoint candidato")
print("--------------------")
print("Rank       :", int(best["break_rank"]))
print("Área       :", round(best["break_area_km2"], 4), "km²")
print("SSE total  :", round(best["sse_total"], 6))

In [ ]:
# 19. Mostrar breakpoint sobre la distribución

break_rank = int(best["break_rank"])

top_n = 50

fig, ax = plt.subplots(figsize=(10, 6))

ax.plot(
    rank_area["rank"].iloc[:top_n],
    rank_area["area_km2"].iloc[:top_n],
    marker="o",
    markersize=4
)

ax.axvline(
    break_rank,
    linestyle="--",
    linewidth=1.5
)

ax.set_yscale("log")
ax.set_xlabel("Rango del componente")
ax.set_ylabel("Área del componente (km², escala log)")
ax.set_title(
    f"Breakpoint candidato — rank {break_rank}"
)

plt.show()

In [ ]:
# 20. Distancia mínima de cada componente al componente principal
# Threshold EMC-BUILT = 10 %

from scipy import ndimage

MAIN_COMPONENT_ID = int(stats_10.iloc[0]["component_id"])

main_mask = labels_10 == MAIN_COMPONENT_ID

# distance_transform_edt calcula distancia al píxel True más próximo
# Se invierte main_mask para obtener distancia desde cualquier píxel
# hasta el componente principal.
distance_to_main_px = ndimage.distance_transform_edt(
    ~main_mask
)

distance_to_main_m = distance_to_main_px * 30.0

component_spatial = []

for _, row in stats_10.head(100).iterrows():

    cid = int(row["component_id"])

    component_mask = labels_10 == cid

    if cid == MAIN_COMPONENT_ID:
        min_distance_m = 0.0
    else:
        min_distance_m = np.min(
            distance_to_main_m[component_mask]
        )

    component_spatial.append({
        "rank": int(row["rank"]),
        "component_id": cid,
        "area_km2": float(row["area_km2"]),
        "min_distance_main_km": min_distance_m / 1000.0
    })

component_spatial = pd.DataFrame(component_spatial)

display(component_spatial.head(30))

In [ ]:
# 21. Área vs distancia al continuo principal

fig, ax = plt.subplots(figsize=(9, 7))

ax.scatter(
    component_spatial["min_distance_main_km"],
    component_spatial["area_km2"]
)

for _, r in component_spatial.head(20).iterrows():
    ax.text(
        r["min_distance_main_km"],
        r["area_km2"],
        str(int(r["rank"])),
        fontsize=9
    )

ax.set_yscale("log")

ax.set_xlabel(
    "Distancia mínima al componente principal (km)"
)
ax.set_ylabel(
    "Área del componente (km², escala log)"
)
ax.set_title(
    "Tamaño y proximidad de componentes built-up ≥ 10 %"
)

plt.show()

In [ ]:
# 22. Tabla ordenada por proximidad espacial

display(
    component_spatial.sort_values(
        "min_distance_main_km"
    ).head(40)
)

In [ ]:
# 23. Construcción provisional de BU_core

CORE_MIN_AREA_KM2 = 0.333

core_ids = stats_10.loc[
    stats_10["area_km2"] >= CORE_MIN_AREA_KM2,
    "component_id"
].astype(int).values

BU_core = np.isin(labels_10, core_ids)

print("Umbral mínimo de componente:", CORE_MIN_AREA_KM2, "km²")
print("Número de componentes core :", len(core_ids))

core_pixels = np.sum(BU_core)
core_area_km2 = core_pixels * 900 / 1e6

print("Área total BU_core         :", round(core_area_km2, 2), "km²")

all_area_km2 = np.sum(mask_10) * 900 / 1e6

print(
    "Fracción del BU>=10% incluida en core:",
    round(100 * core_area_km2 / all_area_km2, 2),
    "%"
)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 7))

axes[0].imshow(mask_10)
axes[0].set_title("BU_all — EMC-BUILT fraction ≥ 10 %")
axes[0].axis("off")

axes[1].imshow(BU_core)
axes[1].set_title(
    f"BU_core — componentes ≥ {CORE_MIN_AREA_KM2:.3f} km²"
)
axes[1].axis("off")

plt.tight_layout()
plt.show()

In [ ]:
# Distancia euclidiana desde cada píxel hasta BU_core

distance_core_px = ndimage.distance_transform_edt(
    ~BU_core
)

distance_core_m = distance_core_px * 30.0
distance_core_km = distance_core_m / 1000.0

print("Distancia mínima:", np.nanmin(distance_core_km), "km")
print("Distancia máxima:", np.nanmax(distance_core_km), "km")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 9))

im = ax.imshow(
    distance_core_km,
    vmin=0
)

ax.set_title(
    "Distancia a agrupaciones construidas sustanciales"
)

ax.axis("off")

plt.colorbar(
    im,
    ax=ax,
    label="Distancia (km)"
)

plt.show()

## 06. Bandas periféricas y soporte TEST fijo


In [ ]:
bands = {
    "B1_0_2km":  (distance_core_km > 0) & (distance_core_km <= 2),
    "B2_2_4km":  (distance_core_km > 2) & (distance_core_km <= 4),
    "B3_4_6km":  (distance_core_km > 4) & (distance_core_km <= 6),
    "B4_6_8km":  (distance_core_km > 6) & (distance_core_km <= 8),
    "B5_8_10km": (distance_core_km > 8) & (distance_core_km <= 10),
}

In [ ]:
print("Superficie bruta por banda")
print("--------------------------")

for name, band in bands.items():

    n = np.sum(band)
    area_km2 = n * 900 / 1e6

    print(
        f"{name:12s}: "
        f"{n:8,d} píxeles | "
        f"{area_km2:7.2f} km²"
    )

In [ ]:
print("\nSuperficie después de excluir BU_all")
print("-------------------------------------")

bands_nonbuilt = {}

for name, band in bands.items():

    clean = band & (~mask_10)

    bands_nonbuilt[name] = clean

    n = np.sum(clean)
    area_km2 = n * 900 / 1e6

    print(
        f"{name:12s}: "
        f"{n:8,d} píxeles | "
        f"{area_km2:7.2f} km²"
    )

In [ ]:
# 27. Inspección de máscaras existentes

QUALITY_DIR = ROOT / "04_quality_masks"

folders_to_check = [
    "final_land_domain",
    "common_valid_mask",
    "hydro_residual_exclusion",
    "land_domain_T30"
]

for folder_name in folders_to_check:

    folder = QUALITY_DIR / folder_name

    print("\n" + "=" * 80)
    print(folder_name)
    print("=" * 80)

    if not folder.exists():
        print("No existe")
        continue

    files = list(folder.rglob("*.tif"))

    if len(files) == 0:
        print("No se encontraron TIFF")
        continue

    for f in files:
        print(f.relative_to(ROOT))

In [ ]:
import rasterio
import numpy as np

for folder_name in folders_to_check:

    folder = QUALITY_DIR / folder_name

    if not folder.exists():
        continue

    for f in folder.rglob("*.tif"):

        print("\n" + "=" * 80)
        print(f.relative_to(ROOT))
        print("=" * 80)

        with rasterio.open(f) as src:

            arr = src.read(1)

            print("CRS        :", src.crs)
            print("Shape      :", src.shape)
            print("Resolution :", src.res)
            print("Bounds     :", src.bounds)
            print("NoData     :", src.nodata)
            print("Dtype      :", src.dtypes[0])

            vals, counts = np.unique(
                arr,
                return_counts=True
            )

            print("Valores únicos:")
            print(list(zip(vals[:20], counts[:20])))

            if len(vals) > 20:
                print("... más valores:", len(vals))

In [ ]:
# 27. Soporte espacial fijo para TEST 2023–2025

LAND_FINAL_PATH = (
    ROOT / "04_quality_masks" /
    "final_land_domain" /
    "land_domain_final_EXP10B_V5.tif"
)

COMMON_VALID_PATHS = {
    2023: ROOT / "04_quality_masks" /
          "common_valid_mask" /
          "EXP10Bv6_common_valid_mask_2023.tif",

    2024: ROOT / "04_quality_masks" /
          "common_valid_mask" /
          "EXP10Bv6_common_valid_mask_2024.tif",

    2025: ROOT / "04_quality_masks" /
          "common_valid_mask" /
          "EXP10Bv6_common_valid_mask_2025.tif",
}

with rasterio.open(LAND_FINAL_PATH) as src:
    land_final = src.read(1) == 1

common_masks = {}

for year, path in COMMON_VALID_PATHS.items():

    with rasterio.open(path) as src:
        common_masks[year] = src.read(1) == 1

    print(
        year,
        f"{common_masks[year].sum():,}",
        "píxeles válidos"
    )

# Intersección espacial común a los tres años TEST
common_test_2023_2025 = (
    common_masks[2023] &
    common_masks[2024] &
    common_masks[2025]
)

# Por seguridad se intersecta también con el dominio terrestre final
fixed_test_support = (
    land_final &
    common_test_2023_2025
)

print("\n--- Soporte fijo TEST ---")
print(
    "Land final:",
    f"{land_final.sum():,}",
    "píxeles"
)

print(
    "Common TEST 2023–2025:",
    f"{common_test_2023_2025.sum():,}",
    "píxeles"
)

print(
    "Soporte fijo final:",
    f"{fixed_test_support.sum():,}",
    "píxeles"
)

print(
    "Área:",
    round(
        fixed_test_support.sum() * 900 / 1e6,
        2
    ),
    "km²"
)

In [ ]:
# 28. Bandas periféricas sobre soporte TEST fijo

bands_fixed = {}

print(
    "Bandas después de aplicar:\n"
    "distancia + soporte TEST fijo + exclusión built-up\n"
)

for name, band in bands.items():

    clean = (
        band &
        fixed_test_support &
        (~mask_10)
    )

    bands_fixed[name] = clean

    n = clean.sum()
    area_km2 = n * 900 / 1e6

    print(
        f"{name:12s}: "
        f"{n:8,d} píxeles | "
        f"{area_km2:7.2f} km²"
    )

In [ ]:
# 29. Bandas extendidas para buscar el plateau térmico

band_edges = [
    (0, 2),
    (2, 4),
    (4, 6),
    (6, 8),
    (8, 10),
    (10, 12),
    (12, 14),
    (14, 16),
]

bands_extended = {}

for lo, hi in band_edges:

    name = f"B_{lo}_{hi}km"

    band = (
        (distance_core_km > lo) &
        (distance_core_km <= hi) &
        fixed_test_support &
        (~mask_10)
    )

    bands_extended[name] = band

    n = band.sum()

    print(
        f"{name:10s}: "
        f"{n:8,d} píxeles | "
        f"{n * 900 / 1e6:7.2f} km²"
    )

In [ ]:
# 30. Buscar LST observada/predicha exportada en 07_models

MODELS_DIR = ROOT / "07_models"

for p in MODELS_DIR.rglob("*.tif"):

    name = p.name.lower()

    if (
        any(str(y) in name for y in [2023, 2024, 2025])
        and "lst" in name
        and ("observ" in name or "predict" in name)
    ):
        print(p.relative_to(ROOT))

In [ ]:
# 30. LST observadas en °C — TEST 2023–2025

RASTERS_C_DIR = (
    ROOT / "07_models" /
    "t3_climate" /
    "module08_independent_2016_2025" /
    "09_physical_unit_evaluation_and_exports" /
    "rasters_celsius"
)

LST_PATHS = {
    2023: RASTERS_C_DIR / "M5_observed_LST_C_2023.tif",
    2024: RASTERS_C_DIR / "M5_observed_LST_C_2024.tif",
    2025: RASTERS_C_DIR / "M5_observed_LST_C_2025.tif",
}

for year, path in LST_PATHS.items():

    print(year, path.exists(), path.name)

In [ ]:
# 31. Verificación de las LST observadas

for year, path in LST_PATHS.items():

    with rasterio.open(path) as src:

        arr = src.read(1).astype("float32")

        valid_lst = np.isfinite(arr)

        if src.nodata is not None:
            valid_lst &= arr != src.nodata

        vals = arr[valid_lst]

        print("\n" + "=" * 70)
        print(year)
        print("=" * 70)

        print("CRS        :", src.crs)
        print("Shape      :", src.shape)
        print("Resolution :", src.res)
        print("Bounds     :", src.bounds)
        print("Transform  :", src.transform == rasterio.open(REF_PATH).transform)
        print("NoData     :", src.nodata)

        print("\nLST °C")
        print("N     :", f"{vals.size:,}")
        print("Min   :", round(float(vals.min()), 3))
        print("P01   :", round(float(np.percentile(vals, 1)), 3))
        print("Median:", round(float(np.median(vals)), 3))
        print("Mean  :", round(float(vals.mean()), 3))
        print("P99   :", round(float(np.percentile(vals, 99)), 3))
        print("Max   :", round(float(vals.max()), 3))

In [ ]:
# 32. Perfil térmico observado por banda periférica

thermal_rows = []

for year, path in LST_PATHS.items():

    with rasterio.open(path) as src:

        lst = src.read(1).astype("float32")
        nodata = src.nodata

    lst_valid = np.isfinite(lst)

    if nodata is not None:
        lst_valid &= lst != nodata

    for name, band in bands_extended.items():

        analysis_mask = (
            band &
            fixed_test_support &
            (~mask_10) &
            lst_valid
        )

        vals = lst[analysis_mask]

        if vals.size == 0:
            continue

        q25 = np.percentile(vals, 25)
        q75 = np.percentile(vals, 75)

        thermal_rows.append({
            "year": year,
            "band": name,
            "n_pixels": int(vals.size),
            "area_km2": vals.size * 900 / 1e6,
            "mean_C": float(np.mean(vals)),
            "median_C": float(np.median(vals)),
            "p25_C": float(q25),
            "p75_C": float(q75),
            "iqr_C": float(q75 - q25),
            "std_C": float(np.std(vals))
        })

thermal_stats = pd.DataFrame(thermal_rows)

display(
    thermal_stats.round({
        "area_km2": 2,
        "mean_C": 3,
        "median_C": 3,
        "p25_C": 3,
        "p75_C": 3,
        "iqr_C": 3,
        "std_C": 3
    })
)

In [ ]:
# 33. Cambio de la mediana entre bandas consecutivas

band_order = [
    "B_0_2km",
    "B_2_4km",
    "B_4_6km",
    "B_6_8km",
    "B_8_10km"
]

gradient_rows = []

for year in [2023, 2024, 2025]:

    d = (
        thermal_stats[
            thermal_stats["year"] == year
        ]
        .set_index("band")
        .reindex(band_order)
    )

    for i in range(len(band_order) - 1):

        b1 = band_order[i]
        b2 = band_order[i + 1]

        if (
            pd.notna(d.loc[b1, "median_C"])
            and pd.notna(d.loc[b2, "median_C"])
        ):

            gradient_rows.append({
                "year": year,
                "from_band": b1,
                "to_band": b2,
                "delta_median_C": (
                    d.loc[b2, "median_C"] -
                    d.loc[b1, "median_C"]
                )
            })

thermal_gradient = pd.DataFrame(gradient_rows)

display(
    thermal_gradient.round(3)
)

In [ ]:
# 34. Gradiente térmico periférico observado

band_midpoints = {
    "B_0_2km": 1,
    "B_2_4km": 3,
    "B_4_6km": 5,
    "B_6_8km": 7,
    "B_8_10km": 9
}

plot_df = thermal_stats[
    thermal_stats["band"].isin(band_order)
].copy()

plot_df["distance_mid_km"] = (
    plot_df["band"].map(band_midpoints)
)

fig, ax = plt.subplots(figsize=(9, 6))

for year in [2023, 2024, 2025]:

    d = (
        plot_df[
            plot_df["year"] == year
        ]
        .sort_values("distance_mid_km")
    )

    ax.plot(
        d["distance_mid_km"],
        d["median_C"],
        marker="o",
        label=str(year)
    )

ax.set_xlabel("Distance to BU_core (km)")
ax.set_ylabel("Observed median LST (°C)")
ax.set_title("Peripheral thermal gradient — TEST 2023–2025")

ax.set_xticks([1, 3, 5, 7, 9])
ax.set_xticklabels([
    "0–2",
    "2–4",
    "4–6",
    "6–8",
    "8–10"
])

ax.legend(title="Year")

plt.show()

## 07. Control espectral sobre la grilla canónica


In [ ]:
# 35. Índices espectrales observados — TEST 2023–2025

RAW_V3_DIR = (
    ROOT /
    "03_annual_raw_composites" /
    "v3_expanded_raw"
)

SPECTRAL_INDICES = ["NDVI", "NDBI", "NDMI"]

INDEX_PATHS = {
    year: {
        index: RAW_V3_DIR /
        f"EXP10Bv3_{index}_median_{year}.tif"
        for index in SPECTRAL_INDICES
    }
    for year in [2023, 2024, 2025]
}

for year in [2023, 2024, 2025]:
    print(f"\n{year}")

    for index in SPECTRAL_INDICES:
        path = INDEX_PATHS[year][index]

        print(
            index,
            path.exists(),
            path.name
        )

In [ ]:
# 36. Auditoría geométrica de los índices

with rasterio.open(REF_PATH) as ref:
    ref_transform = ref.transform
    ref_shape = ref.shape
    ref_crs = ref.crs

for year in [2023, 2024, 2025]:

    for index in SPECTRAL_INDICES:

        path = INDEX_PATHS[year][index]

        with rasterio.open(path) as src:

            print(
                year,
                index,
                "| CRS:", src.crs == ref_crs,
                "| Shape:", src.shape == ref_shape,
                "| Transform:", src.transform == ref_transform,
                "| NoData:", src.nodata
            )

In [ ]:
# 36B. Diagnóstico exacto de la diferencia geométrica

with rasterio.open(REF_PATH) as ref:
    print("REFERENCIA")
    print("Shape :", ref.shape)
    print("Bounds:", ref.bounds)
    print("Transform:", ref.transform)

for year in [2023, 2024, 2025]:
    for index in SPECTRAL_INDICES:

        path = INDEX_PATHS[year][index]

        with rasterio.open(path) as src:
            print(
                f"\n{year} {index}",
                "\nShape :", src.shape,
                "\nBounds:", src.bounds,
                "\nRes   :", src.res,
                "\nTransform igual:", src.transform == ref.transform
            )

In [ ]:
# 36C. Lectura de productos V3 sobre la grilla canónica
# Sin reproyección ni remuestreo

from rasterio.windows import from_bounds

def read_on_reference_grid(path, ref_path):

    with rasterio.open(ref_path) as ref:
        ref_bounds = ref.bounds
        ref_crs = ref.crs
        ref_shape = ref.shape
        ref_transform = ref.transform

    with rasterio.open(path) as src:

        if src.crs != ref_crs:
            raise ValueError(
                f"CRS incompatible: {src.crs} != {ref_crs}"
            )

        if src.res != (30.0, 30.0):
            raise ValueError(
                f"Resolución inesperada: {src.res}"
            )

        # Ventana correspondiente exactamente al dominio canónico
        window = from_bounds(
            *ref_bounds,
            transform=src.transform
        )

        window = window.round_offsets().round_lengths()

        arr = src.read(
            1,
            window=window
        ).astype("float32")

        window_transform = src.window_transform(window)

    if arr.shape != ref_shape:
        raise ValueError(
            f"Shape tras recorte: {arr.shape}; esperado {ref_shape}"
        )

    if window_transform != ref_transform:
        raise ValueError(
            "El transform tras el recorte no coincide con la grilla canónica."
        )

    return arr

In [ ]:
# 37. Diagnóstico espectral por banda
# Productos V3 recortados en memoria a la grilla canónica

spectral_bands = {
    "2–4 km": bands_fixed["B2_2_4km"],
    "4–6 km": bands_fixed["B3_4_6km"],
    "6–8 km": bands_fixed["B4_6_8km"],
}

spectral_rows = []

for year in [2023, 2024, 2025]:

    for index in SPECTRAL_INDICES:

        path = INDEX_PATHS[year][index]

        arr = read_on_reference_grid(
            path,
            REF_PATH
        )

        # Los productos V3 no declaran NoData.
        # La validez espacial se controla con fixed_test_support.
        arr_valid = np.isfinite(arr)

        for band_name, band_mask in spectral_bands.items():

            m = (
                band_mask &
                fixed_test_support &
                (~mask_10) &
                arr_valid
            )

            vals = arr[m]

            if vals.size == 0:
                continue

            q25 = np.percentile(vals, 25)
            q75 = np.percentile(vals, 75)

            spectral_rows.append({
                "year": year,
                "index": index,
                "band": band_name,
                "n_pixels": int(vals.size),
                "median": float(np.median(vals)),
                "mean": float(np.mean(vals)),
                "p25": float(q25),
                "p75": float(q75),
                "iqr": float(q75 - q25)
            })

spectral_stats = pd.DataFrame(spectral_rows)

display(
    spectral_stats.round({
        "median": 3,
        "mean": 3,
        "p25": 3,
        "p75": 3,
        "iqr": 3
    })
)

## 08. Sensibilidad de `BU_core`, referencia final y temperaturas de referencia


In [ ]:
# 39. Sensibilidad de R al umbral mínimo de área de BU_core

CORE_THRESHOLDS = [0.25, 0.333, 0.50]

sensitivity_rows = []

for amin in CORE_THRESHOLDS:

    ids = stats_10.loc[
        stats_10["area_km2"] >= amin,
        "component_id"
    ].astype(int).values

    core = np.isin(labels_10, ids)

    dist_km = (
        ndimage.distance_transform_edt(~core) * 30.0 / 1000.0
    )

    # Banda de referencia 4–6 km
    R_test = (
        (dist_km > 4) &
        (dist_km <= 6) &
        fixed_test_support &
        (~mask_10)
    )

    row = {
        "core_min_area_km2": amin,
        "n_core_components": len(ids),
        "R_pixels": int(R_test.sum()),
        "R_area_km2": R_test.sum() * 900 / 1e6
    }

    for year, path in LST_PATHS.items():

        with rasterio.open(path) as src:
            lst = src.read(1).astype("float32")
            nodata = src.nodata

        valid_lst = np.isfinite(lst)

        if nodata is not None:
            valid_lst &= lst != nodata

        vals = lst[R_test & valid_lst]

        row[f"Tref_{year}_C"] = float(
            np.median(vals)
        )

    sensitivity_rows.append(row)

sensitivity_R = pd.DataFrame(sensitivity_rows)

display(
    sensitivity_R.round({
        "R_area_km2": 2,
        "Tref_2023_C": 3,
        "Tref_2024_C": 3,
        "Tref_2025_C": 3
    })
)

In [ ]:
# 40. Congelar y exportar la referencia SUHI definitiva

R_final = (
    (distance_core_km > 4.0) &
    (distance_core_km <= 6.0) &
    fixed_test_support &
    (~mask_10)
)

print("Píxeles R :", f"{R_final.sum():,}")
print(
    "Área R    :",
    round(R_final.sum() * 900 / 1e6, 2),
    "km²"
)

print(
    "Píxeles built-up dentro de R:",
    int(np.sum(R_final & mask_10))
)

assert R_final.sum() == 167527
assert np.sum(R_final & mask_10) == 0

In [ ]:
REFERENCE_OUT_DIR = (
    ROOT /
    "01_aoi_reference" /
    "emc_built_2022" /
    "processed"
)

REFERENCE_OUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

R_PATH = (
    REFERENCE_OUT_DIR /
    "SUHI_reference_R_4_6km_EMC_BUILT10_core0333_TEST2023_2025.tif"
)

with rasterio.open(REF_PATH) as ref:

    profile = ref.profile.copy()

    profile.update(
        dtype="uint8",
        count=1,
        nodata=None,
        compress="deflate"
    )

    with rasterio.open(
        R_PATH,
        "w",
        **profile
    ) as dst:

        dst.write(
            R_final.astype("uint8"),
            1
        )

print("✓ Referencia SUHI guardada:")
print(R_PATH)

In [ ]:
PRED_PATHS = {
    2023: RASTERS_C_DIR / "M5_predicted_LST_C_2023.tif",
    2024: RASTERS_C_DIR / "M5_predicted_LST_C_2024.tif",
    2025: RASTERS_C_DIR / "M5_predicted_LST_C_2025.tif",
}

In [ ]:
# 41. Auditoría de soporte de R en observed y predicted

reference_support_rows = []

for year in [2023, 2024, 2025]:

    with rasterio.open(LST_PATHS[year]) as src:
        obs = src.read(1).astype("float32")
        obs_nodata = src.nodata

    with rasterio.open(PRED_PATHS[year]) as src:
        pred = src.read(1).astype("float32")
        pred_nodata = src.nodata

    obs_valid = np.isfinite(obs)
    pred_valid = np.isfinite(pred)

    if obs_nodata is not None:
        obs_valid &= obs != obs_nodata

    if pred_nodata is not None:
        pred_valid &= pred != pred_nodata

    R_obs = R_final & obs_valid
    R_pred = R_final & pred_valid
    R_common = R_final & obs_valid & pred_valid

    reference_support_rows.append({
        "year": year,
        "R_geometry": int(R_final.sum()),
        "R_obs_valid": int(R_obs.sum()),
        "R_pred_valid": int(R_pred.sum()),
        "R_common_valid": int(R_common.sum()),
        "lost_from_R": int(
            R_final.sum() - R_common.sum()
        )
    })

reference_support = pd.DataFrame(
    reference_support_rows
)

display(reference_support)

In [ ]:
# 42. Temperatura de referencia observada y predicha
# sobre exactamente los mismos píxeles de R

reference_rows = []

reference_masks_common = {}

for year in [2023, 2024, 2025]:

    with rasterio.open(LST_PATHS[year]) as src:
        obs = src.read(1).astype("float32")
        obs_nodata = src.nodata

    with rasterio.open(PRED_PATHS[year]) as src:
        pred = src.read(1).astype("float32")
        pred_nodata = src.nodata

    obs_valid = np.isfinite(obs)
    pred_valid = np.isfinite(pred)

    if obs_nodata is not None:
        obs_valid &= obs != obs_nodata

    if pred_nodata is not None:
        pred_valid &= pred != pred_nodata

    R_common = (
        R_final &
        obs_valid &
        pred_valid
    )

    reference_masks_common[year] = R_common

    obs_ref = obs[R_common]
    pred_ref = pred[R_common]

    Tref_obs = float(np.median(obs_ref))
    Tref_pred = float(np.median(pred_ref))

    reference_rows.append({
        "year": year,
        "n_reference": int(R_common.sum()),
        "Tref_obs_C": Tref_obs,
        "Tref_pred_C": Tref_pred,
        "Tref_error_C": Tref_pred - Tref_obs,
        "Tref_abs_error_C": abs(Tref_pred - Tref_obs)
    })

reference_stats = pd.DataFrame(reference_rows)

display(
    reference_stats.round({
        "Tref_obs_C": 3,
        "Tref_pred_C": 3,
        "Tref_error_C": 3,
        "Tref_abs_error_C": 3
    })
)

In [ ]:
# 43. SUHI observado y predicho 2023–2025

SUHI = {}

for year in [2023, 2024, 2025]:

    with rasterio.open(LST_PATHS[year]) as src:
        obs = src.read(1).astype("float32")
        obs_nodata = src.nodata

    with rasterio.open(PRED_PATHS[year]) as src:
        pred = src.read(1).astype("float32")
        pred_nodata = src.nodata

    obs_valid = np.isfinite(obs)
    pred_valid = np.isfinite(pred)

    if obs_nodata is not None:
        obs_valid &= obs != obs_nodata

    if pred_nodata is not None:
        pred_valid &= pred != pred_nodata

    # Soporte común para comparar ambos productos
    common_domain = (
        fixed_test_support &
        obs_valid &
        pred_valid
    )

    row = reference_stats.loc[
        reference_stats["year"] == year
    ].iloc[0]

    Tref_obs = row["Tref_obs_C"]
    Tref_pred = row["Tref_pred_C"]

    suhi_obs = np.full(
        obs.shape,
        np.nan,
        dtype=np.float32
    )

    suhi_pred = np.full(
        pred.shape,
        np.nan,
        dtype=np.float32
    )

    suhi_obs[common_domain] = (
        obs[common_domain] - Tref_obs
    )

    suhi_pred[common_domain] = (
        pred[common_domain] - Tref_pred
    )

    suhi_residual = (
        suhi_pred - suhi_obs
    )

    SUHI[year] = {
        "obs": suhi_obs,
        "pred": suhi_pred,
        "residual": suhi_residual,
        "common_domain": common_domain
    }

    print(
        year,
        "| píxeles SUHI:",
        f"{common_domain.sum():,}"
    )

In [ ]:

# ============================================================
# 08B. Exportar tablas intermedias congeladas
# ============================================================

OUTPUT_DIR = (
    ROOT
    / "03_outputs"
)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

TABLE_EXPORTS = {
    "thermal_stats.csv": thermal_stats,
    "thermal_gradient.csv": thermal_gradient,
    "spectral_stats.csv": spectral_stats,
    "sensitivity_R.csv": sensitivity_R,
    "reference_support.csv": reference_support,
    "reference_stats.csv": reference_stats,
}

for filename, table in TABLE_EXPORTS.items():
    path = OUTPUT_DIR / filename
    table.to_csv(path, index=False)
    print("Guardado:", path)


## 09. Fase A — Auditoría full-domain de M5


In [ ]:
# ============================================================
# FASE A — Auditoría full-domain de M5
# A1. Métricas full-domain 2023–2025
# ============================================================

from pathlib import Path
import numpy as np
import pandas as pd
import rasterio

# ------------------------------------------------------------
# Rutas
# ------------------------------------------------------------

BASE = Path(
    "/content/drive/MyDrive/Barranquilla_LST_2013_2026_Modeling"
)

RASTER_DIR = (
    BASE
    / "07_models"
    / "t3_climate"
    / "module08_independent_2016_2025"
    / "09_physical_unit_evaluation_and_exports"
    / "rasters_celsius"
)

YEARS = [2023, 2024, 2025]


# ------------------------------------------------------------
# Funciones
# ------------------------------------------------------------

def read_raster(path):
    with rasterio.open(path) as src:
        arr = src.read(1).astype("float64")
        nodata = src.nodata

        if nodata is not None:
            arr[arr == nodata] = np.nan

        meta = {
            "crs": src.crs,
            "shape": (src.height, src.width),
            "transform": src.transform,
            "nodata": nodata,
        }

    return arr, meta


def compute_metrics(obs, pred):
    valid = np.isfinite(obs) & np.isfinite(pred)

    y = obs[valid]
    p = pred[valid]

    err = p - y

    rmse = np.sqrt(np.mean(err**2))
    mae = np.mean(np.abs(err))
    bias = np.mean(err)

    ss_res = np.sum((y - p)**2)
    ss_tot = np.sum((y - np.mean(y))**2)
    r2 = 1 - ss_res / ss_tot

    # Auditoría adicional
    slope, intercept = np.polyfit(y, p, 1)

    return {
        "n_common": int(valid.sum()),
        "rmse_c": float(rmse),
        "mae_c": float(mae),
        "bias_c": float(bias),
        "r2": float(r2),
        "slope": float(slope),
        "intercept_c": float(intercept),
    }


# ------------------------------------------------------------
# Evaluación anual
# ------------------------------------------------------------

rows = []

for year in YEARS:

    obs_path = RASTER_DIR / f"M5_observed_LST_C_{year}.tif"
    pred_path = RASTER_DIR / f"M5_predicted_LST_C_{year}.tif"

    obs, obs_meta = read_raster(obs_path)
    pred, pred_meta = read_raster(pred_path)

    # Control geométrico
    assert obs_meta["crs"] == pred_meta["crs"]
    assert obs_meta["shape"] == pred_meta["shape"]
    assert obs_meta["transform"] == pred_meta["transform"]

    metrics = compute_metrics(obs, pred)
    metrics["year"] = year

    rows.append(metrics)


full_domain_metrics = pd.DataFrame(rows)[
    [
        "year",
        "n_common",
        "rmse_c",
        "mae_c",
        "bias_c",
        "r2",
        "slope",
        "intercept_c",
    ]
]

display(full_domain_metrics.round(4))

In [ ]:
# ============================================================
# FASE A — A2. Comparación formal:
# TEST por parches vs full-domain
# ============================================================

from pathlib import Path
import pandas as pd

# ------------------------------------------------------------
# Métricas TEST oficiales de M5
# ------------------------------------------------------------

test_metrics = pd.DataFrame({
    "year":      [2023, 2024, 2025],
    "rmse_test": [1.932, 1.744, 1.758],
    "mae_test":  [1.517, 1.325, 1.354],
    "bias_test": [-0.162, -0.646, -0.084],
    "r2_test":   [0.760, 0.730, 0.760],
})

# ------------------------------------------------------------
# Preparar métricas full-domain calculadas en A1
# ------------------------------------------------------------

fd = full_domain_metrics.rename(columns={
    "rmse_c": "rmse_full",
    "mae_c": "mae_full",
    "bias_c": "bias_full",
    "r2": "r2_full",
}).copy()

# ------------------------------------------------------------
# Unir ambos soportes evaluativos
# ------------------------------------------------------------

comparison = test_metrics.merge(fd, on="year", how="inner")

# Diferencias: full-domain - TEST
comparison["delta_rmse"] = (
    comparison["rmse_full"] - comparison["rmse_test"]
)

comparison["delta_mae"] = (
    comparison["mae_full"] - comparison["mae_test"]
)

comparison["delta_bias"] = (
    comparison["bias_full"] - comparison["bias_test"]
)

comparison["delta_r2"] = (
    comparison["r2_full"] - comparison["r2_test"]
)

# Diferencias relativas solo para RMSE y MAE
comparison["delta_rmse_pct"] = (
    100 * comparison["delta_rmse"] / comparison["rmse_test"]
)

comparison["delta_mae_pct"] = (
    100 * comparison["delta_mae"] / comparison["mae_test"]
)

# ------------------------------------------------------------
# Mostrar
# ------------------------------------------------------------

display(
    comparison[
        [
            "year",
            "n_common",
            "rmse_test",
            "rmse_full",
            "delta_rmse",
            "delta_rmse_pct",
            "mae_test",
            "mae_full",
            "delta_mae",
            "delta_mae_pct",
            "bias_test",
            "bias_full",
            "delta_bias",
            "r2_test",
            "r2_full",
            "delta_r2",
            "slope",
            "intercept_c",
        ]
    ].round(4)
)

# ------------------------------------------------------------
# Guardar resultados
# ------------------------------------------------------------

OUTPUT_DIR = (
    Path("/content/drive/MyDrive/"
         "Barranquilla_LST_2013_2026_Modeling/03_outputs")
)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

full_domain_path = (
    OUTPUT_DIR / "m5_full_domain_metrics_2023_2025.csv"
)

comparison_path = (
    OUTPUT_DIR / "m5_full_domain_vs_test_metrics.csv"
)

full_domain_metrics.to_csv(
    full_domain_path,
    index=False
)

comparison.to_csv(
    comparison_path,
    index=False
)

print("Guardado:")
print(full_domain_path)
print(comparison_path)

## 10. Fase B — Definición final del soporte urbano `U`


In [ ]:
# ============================================================
# FASE B — Comparación BU_all vs BU_core como soporte urbano U
# ============================================================

from pathlib import Path
import numpy as np
import pandas as pd
import rasterio
from scipy import ndimage

# ------------------------------------------------------------
# Rutas
# ------------------------------------------------------------

BASE = Path(
    "/content/drive/MyDrive/Barranquilla_LST_2013_2026_Modeling"
)

EMC_DIR = (
    BASE
    / "01_aoi_reference"
    / "emc_built_2022"
    / "processed"
)

FRACTION_PATH = (
    EMC_DIR
    / "EMC_BUILT_fraction_E2022_EPSG32618_30m.tif"
)

RASTER_DIR = (
    BASE
    / "07_models"
    / "t3_climate"
    / "module08_independent_2016_2025"
    / "09_physical_unit_evaluation_and_exports"
    / "rasters_celsius"
)

R_PATH = (
    EMC_DIR
    / "SUHI_reference_R_4_6km_EMC_BUILT10_core0333_TEST2023_2025.tif"
)

YEARS = [2023, 2024, 2025]

PIXEL_AREA_M2 = 30 * 30
PIXEL_AREA_KM2 = PIXEL_AREA_M2 / 1e6

BU_THRESHOLD = 0.10
CORE_MIN_AREA_KM2 = 0.333


# ------------------------------------------------------------
# Funciones
# ------------------------------------------------------------

def read_raster(path):
    with rasterio.open(path) as src:
        arr = src.read(1).astype("float64")
        nodata = src.nodata

        if nodata is not None:
            arr[arr == nodata] = np.nan

        meta = {
            "crs": src.crs,
            "shape": (src.height, src.width),
            "transform": src.transform,
            "nodata": nodata,
        }

    return arr, meta


# ------------------------------------------------------------
# 1. Leer fracción built-up
# ------------------------------------------------------------

fraction, fraction_meta = read_raster(FRACTION_PATH)

BU_all = (
    np.isfinite(fraction)
    & (fraction >= BU_THRESHOLD)
)

print("BU_all pixels:", int(BU_all.sum()))
print("BU_all area km²:", BU_all.sum() * PIXEL_AREA_KM2)


# ------------------------------------------------------------
# 2. Reconstruir BU_core
#    conectividad de 8 vecinos
# ------------------------------------------------------------

structure = np.ones((3, 3), dtype=int)

labels, n_components = ndimage.label(
    BU_all,
    structure=structure
)

component_pixel_counts = np.bincount(labels.ravel())

# índice 0 = background
component_areas_km2 = component_pixel_counts * PIXEL_AREA_KM2

keep_labels = np.where(
    component_areas_km2 >= CORE_MIN_AREA_KM2
)[0]

# eliminar background
keep_labels = keep_labels[keep_labels != 0]

BU_core = np.isin(labels, keep_labels)

print("\nBU_core components:", len(keep_labels))
print("BU_core pixels:", int(BU_core.sum()))
print("BU_core area km²:", BU_core.sum() * PIXEL_AREA_KM2)


# ------------------------------------------------------------
# 3. Leer máscara de referencia R
# ------------------------------------------------------------

R_arr, R_meta = read_raster(R_PATH)
R_mask = np.isfinite(R_arr) & (R_arr > 0)

print("\nReference R pixels:", int(R_mask.sum()))
print("Reference R area km²:", R_mask.sum() * PIXEL_AREA_KM2)


# ------------------------------------------------------------
# 4. Auditoría geométrica
# ------------------------------------------------------------

assert fraction_meta["shape"] == R_meta["shape"]
assert fraction_meta["transform"] == R_meta["transform"]
assert fraction_meta["crs"] == R_meta["crs"]


# ------------------------------------------------------------
# 5. Comparar BU_all y BU_core por año
# ------------------------------------------------------------

rows = []

for year in YEARS:

    obs_path = RASTER_DIR / f"M5_observed_LST_C_{year}.tif"
    pred_path = RASTER_DIR / f"M5_predicted_LST_C_{year}.tif"

    obs, obs_meta = read_raster(obs_path)
    pred, pred_meta = read_raster(pred_path)

    # Auditoría geométrica
    assert obs_meta["shape"] == fraction_meta["shape"]
    assert pred_meta["shape"] == fraction_meta["shape"]
    assert obs_meta["transform"] == fraction_meta["transform"]
    assert pred_meta["transform"] == fraction_meta["transform"]

    common = (
        np.isfinite(obs)
        & np.isfinite(pred)
    )

    # Referencia anual común
    ref_valid = R_mask & common

    tref_obs = float(np.median(obs[ref_valid]))
    tref_pred = float(np.median(pred[ref_valid]))

    for support_name, support_mask in [
        ("BU_all", BU_all),
        ("BU_core", BU_core),
    ]:

        urban_valid = support_mask & common

        n = int(urban_valid.sum())
        area_km2 = n * PIXEL_AREA_KM2

        urban_obs = float(
            np.median(obs[urban_valid])
        )

        urban_pred = float(
            np.median(pred[urban_valid])
        )

        delta_obs = urban_obs - tref_obs
        delta_pred = urban_pred - tref_pred
        delta_error = delta_pred - delta_obs

        rows.append({
            "year": year,
            "urban_support": support_name,
            "n_pixels": n,
            "area_km2": area_km2,
            "tref_obs_c": tref_obs,
            "tref_pred_c": tref_pred,
            "urban_median_obs_c": urban_obs,
            "urban_median_pred_c": urban_pred,
            "deltaT_obs_c": delta_obs,
            "deltaT_pred_c": delta_pred,
            "deltaT_error_c": delta_error,
        })


urban_support_comparison = pd.DataFrame(rows)

display(
    urban_support_comparison.round(4)
)

In [ ]:
# ============================================================
# FASE B — B2. Guardar comparación y congelar soporte urbano U
# ============================================================

OUTPUT_DIR = (
    Path("/content/drive/MyDrive/"
         "Barranquilla_LST_2013_2026_Modeling/03_outputs")
)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

urban_support_path = (
    OUTPUT_DIR
    / "urban_support_comparison_BUall_vs_BUcore.csv"
)

urban_support_comparison.to_csv(
    urban_support_path,
    index=False
)

print("Guardado:")
print(urban_support_path)

print("\nDECISIÓN FINAL FASE B")
print("--------------------")
print("Urban support U = BU_core ∩ M_common,Y")
print("BU_core components =", len(keep_labels))
print("BU_core pixels     =", int(BU_core.sum()))
print(
    "BU_core area km²  =",
    round(BU_core.sum() * PIXEL_AREA_KM2, 4)
)

## 11. Fase C — Definición final del P95 urbano de SUHI


In [ ]:
# ============================================================
# FASE C — P95 SUHI: dominio completo vs soporte urbano
# ============================================================

rows_p95 = []

for year in YEARS:

    obs_path = RASTER_DIR / f"M5_observed_LST_C_{year}.tif"
    pred_path = RASTER_DIR / f"M5_predicted_LST_C_{year}.tif"

    obs, _ = read_raster(obs_path)
    pred, _ = read_raster(pred_path)

    common = (
        np.isfinite(obs)
        & np.isfinite(pred)
    )

    # --------------------------------------------------------
    # Referencia anual
    # --------------------------------------------------------

    ref_valid = R_mask & common

    tref_obs = float(
        np.median(obs[ref_valid])
    )

    tref_pred = float(
        np.median(pred[ref_valid])
    )

    # --------------------------------------------------------
    # Campos SUHI
    # --------------------------------------------------------

    suhi_obs = obs - tref_obs
    suhi_pred = pred - tref_pred

    # --------------------------------------------------------
    # P95: dominio común completo
    # --------------------------------------------------------

    p95_domain_obs = float(
        np.percentile(
            suhi_obs[common],
            95
        )
    )

    p95_domain_pred = float(
        np.percentile(
            suhi_pred[common],
            95
        )
    )

    # --------------------------------------------------------
    # P95: soporte urbano final U = BU_core
    # --------------------------------------------------------

    urban_valid = BU_core & common

    p95_urban_obs = float(
        np.percentile(
            suhi_obs[urban_valid],
            95
        )
    )

    p95_urban_pred = float(
        np.percentile(
            suhi_pred[urban_valid],
            95
        )
    )

    rows_p95.append({
        "year": year,

        "p95_domain_obs_c": p95_domain_obs,
        "p95_domain_pred_c": p95_domain_pred,
        "p95_domain_error_c":
            p95_domain_pred - p95_domain_obs,

        "p95_urban_obs_c": p95_urban_obs,
        "p95_urban_pred_c": p95_urban_pred,
        "p95_urban_error_c":
            p95_urban_pred - p95_urban_obs,

        "n_domain": int(common.sum()),
        "n_urban": int(urban_valid.sum()),
    })


suhi_p95_comparison = pd.DataFrame(rows_p95)

display(
    suhi_p95_comparison.round(4)
)

In [ ]:
# ============================================================
# FASE C — C2. Guardar comparación y congelar P95 final
# ============================================================

OUTPUT_DIR = (
    Path("/content/drive/MyDrive/"
         "Barranquilla_LST_2013_2026_Modeling/03_outputs")
)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

p95_path = (
    OUTPUT_DIR
    / "suhi_p95_domain_vs_urban.csv"
)

suhi_p95_comparison.to_csv(
    p95_path,
    index=False
)

print("Guardado:")
print(p95_path)

print("\nDECISIÓN FINAL FASE C")
print("--------------------")
print("Primary P95 = P95 of SUHI within U = BU_core")
print("P95_domain retained only as secondary diagnostic.")

## 12. Fase D — Sensibilidad del diagnóstico SUHI


In [ ]:
# ============================================================
# FASE D — Sensibilidad SUHI a la definición de BU_core
#
# Evalúa:
#   A_min = 0.25, 0.333, 0.50 km²
#
# Para cada configuración:
#   - reconstruye BU_core
#   - recalcula distancia
#   - reconstruye R = 4–6 km
#   - calcula Tref obs/pred
#   - calcula DeltaT urbano-referencia
#   - calcula P95 urbano de SUHI
#
# NO guarda archivos todavía.
# ============================================================

from pathlib import Path
import numpy as np
import pandas as pd
import rasterio
from scipy import ndimage


# ------------------------------------------------------------
# 1. Rutas
# ------------------------------------------------------------

BASE = Path(
    "/content/drive/MyDrive/Barranquilla_LST_2013_2026_Modeling"
)

EMC_DIR = (
    BASE
    / "01_aoi_reference"
    / "emc_built_2022"
    / "processed"
)

FRACTION_PATH = (
    EMC_DIR
    / "EMC_BUILT_fraction_E2022_EPSG32618_30m.tif"
)

MASK_DIR = (
    BASE
    / "04_quality_masks"
    / "common_valid_mask"
)

RASTER_DIR = (
    BASE
    / "07_models"
    / "t3_climate"
    / "module08_independent_2016_2025"
    / "09_physical_unit_evaluation_and_exports"
    / "rasters_celsius"
)

YEARS = [2023, 2024, 2025]

A_MIN_VALUES = [0.250, 0.333, 0.500]

BU_THRESHOLD = 0.10

PIXEL_SIZE_M = 30.0
PIXEL_SIZE_KM = PIXEL_SIZE_M / 1000.0
PIXEL_AREA_KM2 = (PIXEL_SIZE_M * PIXEL_SIZE_M) / 1e6


# ------------------------------------------------------------
# 2. Funciones
# ------------------------------------------------------------

def read_raster(path):
    with rasterio.open(path) as src:
        arr = src.read(1).astype("float64")
        nodata = src.nodata

        if nodata is not None:
            arr[arr == nodata] = np.nan

        meta = {
            "crs": src.crs,
            "shape": (src.height, src.width),
            "transform": src.transform,
            "nodata": nodata,
        }

    return arr, meta


def read_binary_mask(path):
    with rasterio.open(path) as src:
        arr = src.read(1)

        meta = {
            "crs": src.crs,
            "shape": (src.height, src.width),
            "transform": src.transform,
        }

    return arr > 0, meta


# ------------------------------------------------------------
# 3. Leer fracción built-up
# ------------------------------------------------------------

fraction, fraction_meta = read_raster(FRACTION_PATH)

BU_all = (
    np.isfinite(fraction)
    & (fraction >= BU_THRESHOLD)
)

print("BU_all")
print("-------")
print("Pixels :", int(BU_all.sum()))
print("Area   :", round(BU_all.sum() * PIXEL_AREA_KM2, 4), "km²")


# ------------------------------------------------------------
# 4. Construir soporte TEST fijo 2023–2025
#
# M_TEST = M_2023 ∩ M_2024 ∩ M_2025
#
# La trazabilidad ya verificó que esta intersección queda
# contenida en la máscara terrestre final.
# ------------------------------------------------------------

test_masks = []

for year in YEARS:

    mask_path = (
        MASK_DIR
        / f"EXP10Bv6_common_valid_mask_{year}.tif"
    )

    mask, mask_meta = read_binary_mask(mask_path)

    assert mask_meta["shape"] == fraction_meta["shape"]
    assert mask_meta["transform"] == fraction_meta["transform"]
    assert mask_meta["crs"] == fraction_meta["crs"]

    test_masks.append(mask)


fixed_test_support = np.logical_and.reduce(test_masks)

print("\nFixed TEST support")
print("------------------")
print("Pixels :", int(fixed_test_support.sum()))
print(
    "Area   :",
    round(fixed_test_support.sum() * PIXEL_AREA_KM2, 4),
    "km²"
)


# ------------------------------------------------------------
# 5. Leer LST observada y predicha una sola vez
# ------------------------------------------------------------

lst_data = {}

for year in YEARS:

    obs_path = (
        RASTER_DIR
        / f"M5_observed_LST_C_{year}.tif"
    )

    pred_path = (
        RASTER_DIR
        / f"M5_predicted_LST_C_{year}.tif"
    )

    obs, obs_meta = read_raster(obs_path)
    pred, pred_meta = read_raster(pred_path)

    assert obs_meta["shape"] == fraction_meta["shape"]
    assert pred_meta["shape"] == fraction_meta["shape"]

    assert obs_meta["transform"] == fraction_meta["transform"]
    assert pred_meta["transform"] == fraction_meta["transform"]

    assert obs_meta["crs"] == fraction_meta["crs"]
    assert pred_meta["crs"] == fraction_meta["crs"]

    lst_data[year] = {
        "obs": obs,
        "pred": pred,
    }


# ------------------------------------------------------------
# 6. Componentes conectados de BU_all
# ------------------------------------------------------------

structure = np.ones((3, 3), dtype=int)

labels, n_components_total = ndimage.label(
    BU_all,
    structure=structure
)

component_pixel_counts = np.bincount(
    labels.ravel()
)

component_areas_km2 = (
    component_pixel_counts
    * PIXEL_AREA_KM2
)


# ------------------------------------------------------------
# 7. Sensibilidad
# ------------------------------------------------------------

rows = []

for a_min in A_MIN_VALUES:

    # --------------------------------------------------------
    # 7.1 BU_core para este A_min
    # --------------------------------------------------------

    keep_labels = np.where(
        component_areas_km2 >= a_min
    )[0]

    # eliminar background = 0
    keep_labels = keep_labels[
        keep_labels != 0
    ]

    BU_core_sens = np.isin(
        labels,
        keep_labels
    )

    n_core_components = len(keep_labels)

    n_core_pixels = int(
        BU_core_sens.sum()
    )

    core_area_km2 = (
        n_core_pixels
        * PIXEL_AREA_KM2
    )


    # --------------------------------------------------------
    # 7.2 Distancia euclidiana a BU_core
    #
    # distance_transform_edt calcula distancia hasta
    # el cero más próximo.
    #
    # ~BU_core:
    #   core    = False = 0
    #   outside = True  = 1
    # --------------------------------------------------------

    distance_km = (
        ndimage.distance_transform_edt(
            ~BU_core_sens
        )
        * PIXEL_SIZE_KM
    )


    # --------------------------------------------------------
    # 7.3 Referencia R = 4–6 km
    #
    # Se mantiene:
    # - soporte TEST fijo
    # - exclusión BU_all
    # --------------------------------------------------------

    R_sens = (
        (distance_km > 4.0)
        & (distance_km <= 6.0)
        & fixed_test_support
        & (~BU_all)
    )

    n_R = int(R_sens.sum())
    area_R_km2 = (
        n_R
        * PIXEL_AREA_KM2
    )


    # --------------------------------------------------------
    # 7.4 Evaluación anual
    # --------------------------------------------------------

    for year in YEARS:

        obs = lst_data[year]["obs"]
        pred = lst_data[year]["pred"]

        common = (
            np.isfinite(obs)
            & np.isfinite(pred)
        )

        # ----------------------------------------------------
        # Referencia térmica anual
        # ----------------------------------------------------

        ref_valid = (
            R_sens
            & common
        )

        if ref_valid.sum() == 0:
            raise RuntimeError(
                f"R sin píxeles válidos: "
                f"A_min={a_min}, year={year}"
            )

        tref_obs = float(
            np.median(obs[ref_valid])
        )

        tref_pred = float(
            np.median(pred[ref_valid])
        )


        # ----------------------------------------------------
        # Soporte urbano:
        # U = BU_core(A_min) ∩ common
        # ----------------------------------------------------

        urban_valid = (
            BU_core_sens
            & common
        )

        if urban_valid.sum() == 0:
            raise RuntimeError(
                f"U sin píxeles válidos: "
                f"A_min={a_min}, year={year}"
            )

        urban_obs = float(
            np.median(
                obs[urban_valid]
            )
        )

        urban_pred = float(
            np.median(
                pred[urban_valid]
            )
        )


        # ----------------------------------------------------
        # Contraste urbano–referencia
        # ----------------------------------------------------

        delta_obs = (
            urban_obs
            - tref_obs
        )

        delta_pred = (
            urban_pred
            - tref_pred
        )

        delta_error = (
            delta_pred
            - delta_obs
        )


        # ----------------------------------------------------
        # SUHI
        # ----------------------------------------------------

        suhi_obs = (
            obs
            - tref_obs
        )

        suhi_pred = (
            pred
            - tref_pred
        )


        # ----------------------------------------------------
        # P95 URBANO
        # definido en la Fase C
        # ----------------------------------------------------

        p95_urban_obs = float(
            np.percentile(
                suhi_obs[urban_valid],
                95
            )
        )

        p95_urban_pred = float(
            np.percentile(
                suhi_pred[urban_valid],
                95
            )
        )

        p95_urban_error = (
            p95_urban_pred
            - p95_urban_obs
        )


        # ----------------------------------------------------
        # Guardar fila
        # ----------------------------------------------------

        rows.append({

            "A_min_km2": a_min,

            "n_core_components":
                n_core_components,

            "n_core_pixels":
                n_core_pixels,

            "core_area_km2":
                core_area_km2,

            "n_R_geometry":
                n_R,

            "R_area_km2":
                area_R_km2,

            "year":
                year,

            "n_R_valid":
                int(ref_valid.sum()),

            "n_U_valid":
                int(urban_valid.sum()),

            "tref_obs_c":
                tref_obs,

            "tref_pred_c":
                tref_pred,

            "tref_error_c":
                tref_pred - tref_obs,

            "urban_median_obs_c":
                urban_obs,

            "urban_median_pred_c":
                urban_pred,

            "deltaT_obs_c":
                delta_obs,

            "deltaT_pred_c":
                delta_pred,

            "deltaT_error_c":
                delta_error,

            "p95_urban_obs_c":
                p95_urban_obs,

            "p95_urban_pred_c":
                p95_urban_pred,

            "p95_urban_error_c":
                p95_urban_error,
        })


# ------------------------------------------------------------
# 8. Tabla completa
# ------------------------------------------------------------

sensitivity_suhi = pd.DataFrame(rows)

display(
    sensitivity_suhi.round(4)
)


# ------------------------------------------------------------
# 9. Tabla compacta para interpretación
# ------------------------------------------------------------

sensitivity_compact = sensitivity_suhi[
    [
        "A_min_km2",
        "n_core_components",
        "n_R_geometry",
        "year",
        "tref_obs_c",
        "deltaT_obs_c",
        "deltaT_pred_c",
        "deltaT_error_c",
        "p95_urban_obs_c",
        "p95_urban_pred_c",
        "p95_urban_error_c",
    ]
].copy()

print("\nRESUMEN PARA DECISIÓN")
print("=====================")

display(
    sensitivity_compact.round(4)
)


# ------------------------------------------------------------
# 10. Control nominal A_min = 0.333 km²
#
# Estos valores deben reproducir aproximadamente:
#   componentes = 18
#   R geometry  = 167527 pixels
# ------------------------------------------------------------

nominal = sensitivity_suhi[
    np.isclose(
        sensitivity_suhi["A_min_km2"],
        0.333
    )
]

nominal_components = int(
    nominal["n_core_components"].iloc[0]
)

nominal_R_pixels = int(
    nominal["n_R_geometry"].iloc[0]
)

print("\nCONTROL CONFIGURACIÓN NOMINAL")
print("=============================")

print(
    "BU_core components:",
    nominal_components,
    "(esperado: 18)"
)

print(
    "R pixels:",
    nominal_R_pixels,
    "(esperado: 167527)"
)

if (
    nominal_components == 18
    and nominal_R_pixels == 167527
):
    print(
        "\nCONTROL NOMINAL: OK"
    )
else:
    print(
        "\nATENCIÓN: la configuración nominal "
        "no reproduce exactamente la geometría esperada."
    )

In [ ]:
# ============================================================
# FASE D — D2. Guardar sensibilidad SUHI
# ============================================================

OUTPUT_DIR = (
    Path("/content/drive/MyDrive/"
         "Barranquilla_LST_2013_2026_Modeling/03_outputs")
)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

sensitivity_path = (
    OUTPUT_DIR
    / "sensitivity_suhi_Amin_025_0333_050.csv"
)

sensitivity_suhi.to_csv(
    sensitivity_path,
    index=False
)

print("Guardado:")
print(sensitivity_path)

print("\nFASE D CERRADA")
print("---------------")
print("Configuración nominal A_min = 0.333 km²")
print("Interpretación robusta frente a 0.25–0.50 km²")

## 13. Fase E — Descomposición del error urbano/referencia


In [ ]:
# ============================================================
# FASE E — Descomposición formal del error SUHI
#
# EU     = error térmico urbano
# ER     = error de la referencia
# EΔT    = EU - ER
# ============================================================

nominal = sensitivity_suhi[
    np.isclose(
        sensitivity_suhi["A_min_km2"],
        0.333
    )
].copy()

error_decomposition = pd.DataFrame({

    "year":
        nominal["year"].values,

    "urban_obs_c":
        nominal["urban_median_obs_c"].values,

    "urban_pred_c":
        nominal["urban_median_pred_c"].values,

    "E_U_c":
        (
            nominal["urban_median_pred_c"]
            - nominal["urban_median_obs_c"]
        ).values,

    "tref_obs_c":
        nominal["tref_obs_c"].values,

    "tref_pred_c":
        nominal["tref_pred_c"].values,

    "E_R_c":
        (
            nominal["tref_pred_c"]
            - nominal["tref_obs_c"]
        ).values,

    "deltaT_obs_c":
        nominal["deltaT_obs_c"].values,

    "deltaT_pred_c":
        nominal["deltaT_pred_c"].values,

    "E_deltaT_c":
        (
            nominal["deltaT_pred_c"]
            - nominal["deltaT_obs_c"]
        ).values,
})

# ------------------------------------------------------------
# Verificación de identidad:
#
# E_deltaT = E_U - E_R
# ------------------------------------------------------------

error_decomposition["EU_minus_ER_c"] = (
    error_decomposition["E_U_c"]
    - error_decomposition["E_R_c"]
)

error_decomposition["identity_error"] = (
    error_decomposition["E_deltaT_c"]
    - error_decomposition["EU_minus_ER_c"]
)

display(
    error_decomposition.round(4)
)

print(
    "\nMáximo error de identidad:",
    np.abs(
        error_decomposition["identity_error"]
    ).max()
)

In [ ]:
# ============================================================
# FASE E — E2. Guardar descomposición del error
# ============================================================

from pathlib import Path

OUTPUT_DIR = (
    Path("/content/drive/MyDrive/"
         "Barranquilla_LST_2013_2026_Modeling/03_outputs")
)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

error_path = (
    OUTPUT_DIR
    / "suhi_error_decomposition_2023_2025.csv"
)

error_decomposition.to_csv(
    error_path,
    index=False
)

print("Guardado:")
print(error_path)

print("\nFASE E CERRADA")
print("---------------")
print("Identidad verificada: E_deltaT = E_U - E_R")
print(
    "Máximo error numérico:",
    np.abs(error_decomposition["identity_error"]).max()
)

## 14. Fase F — Auditoría, exportación y congelación de productos


In [ ]:
# ============================================================
# FASE F — F1. Auditoría física de la máscara R definitiva
# ============================================================

from pathlib import Path
import numpy as np
import rasterio

BASE = Path(
    "/content/drive/MyDrive/Barranquilla_LST_2013_2026_Modeling"
)

R_PATH = (
    BASE
    / "01_aoi_reference"
    / "emc_built_2022"
    / "processed"
    / "SUHI_reference_R_4_6km_EMC_BUILT10_core0333_TEST2023_2025.tif"
)

EXPECTED_SHAPE = (1115, 1257)
EXPECTED_PIXELS = 167527
PIXEL_AREA_KM2 = 0.0009

print("Archivo:")
print(R_PATH)
print("\nExiste:", R_PATH.exists())

with rasterio.open(R_PATH) as src:

    arr = src.read(1)

    print("\nCRS       :", src.crs)
    print("Shape     :", src.shape)
    print("Resolution:", src.res)
    print("NoData    :", src.nodata)
    print("Dtype     :", src.dtypes[0])
    print("Transform :", src.transform)
    print("Bounds    :", src.bounds)

    values, counts = np.unique(
        arr,
        return_counts=True
    )

    print("\nValores únicos:")
    for v, n in zip(values, counts):
        print(v, "->", n)

    R_disk = arr > 0

    n_R = int(R_disk.sum())
    area_R = n_R * PIXEL_AREA_KM2

    print("\nR pixels  :", n_R)
    print("R area km²:", round(area_R, 4))

    assert src.crs.to_epsg() == 32618
    assert src.shape == EXPECTED_SHAPE
    assert src.res == (30.0, 30.0)
    assert n_R == EXPECTED_PIXELS

print("\nAUDITORÍA R: OK")

In [ ]:
# ============================================================
# FASE F0 — Verificación definitiva del soporte SUHI
# ============================================================

rows_support = []

for year in YEARS:

    obs = lst_data[year]["obs"]
    pred = lst_data[year]["pred"]

    finite_common = (
        np.isfinite(obs)
        & np.isfinite(pred)
    )

    suhi_common = (
        fixed_test_support
        & finite_common
    )

    # Soporte urbano final
    urban_final = (
        BU_core
        & suhi_common
    )

    # Referencia anual
    ref_final = (
        R_disk
        & suhi_common
    )

    tref_obs = float(
        np.median(obs[ref_final])
    )

    tref_pred = float(
        np.median(pred[ref_final])
    )

    suhi_obs = obs - tref_obs
    suhi_pred = pred - tref_pred

    delta_obs = float(
        np.median(suhi_obs[urban_final])
    )

    delta_pred = float(
        np.median(suhi_pred[urban_final])
    )

    p95_urban_obs = float(
        np.percentile(
            suhi_obs[urban_final],
            95
        )
    )

    p95_urban_pred = float(
        np.percentile(
            suhi_pred[urban_final],
            95
        )
    )

    rows_support.append({
        "year": year,
        "n_finite_common": int(finite_common.sum()),
        "n_suhi_common": int(suhi_common.sum()),
        "n_reference": int(ref_final.sum()),
        "n_urban": int(urban_final.sum()),
        "tref_obs_c": tref_obs,
        "tref_pred_c": tref_pred,
        "deltaT_obs_c": delta_obs,
        "deltaT_pred_c": delta_pred,
        "p95_urban_obs_c": p95_urban_obs,
        "p95_urban_pred_c": p95_urban_pred,
    })

support_audit = pd.DataFrame(rows_support)

display(
    support_audit.round(4)
)

In [ ]:
# ============================================================
# FASE F — F2. Exportación definitiva de SUHI 2023–2025
#
# Produce:
#   observed
#   predicted
#   residual = predicted - observed
#
# para 2023, 2024 y 2025.
# ============================================================

from pathlib import Path
import numpy as np
import pandas as pd
import rasterio


# ------------------------------------------------------------
# 1. Directorio de salida
# ------------------------------------------------------------

OUTPUT_DIR = (
    Path("/content/drive/MyDrive/"
         "Barranquilla_LST_2013_2026_Modeling/03_outputs")
)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

NODATA_OUT = -9999.0


# Conteos esperados del soporte SUHI definitivo
EXPECTED_VALID = {
    2023: 904942,
    2024: 905134,
    2025: 905276,
}


# ------------------------------------------------------------
# 2. Exportación anual
# ------------------------------------------------------------

export_rows = []

for year in YEARS:

    obs_path = (
        RASTER_DIR
        / f"M5_observed_LST_C_{year}.tif"
    )

    pred_path = (
        RASTER_DIR
        / f"M5_predicted_LST_C_{year}.tif"
    )

    # Leer nuevamente para que la exportación no dependa
    # de arrays modificados accidentalmente en memoria.
    with rasterio.open(obs_path) as src:
        obs = src.read(1).astype("float64")

        obs_nodata = src.nodata

        profile = src.profile.copy()

        if obs_nodata is not None:
            obs[obs == obs_nodata] = np.nan

    with rasterio.open(pred_path) as src:
        pred = src.read(1).astype("float64")

        pred_nodata = src.nodata

        if pred_nodata is not None:
            pred[pred == pred_nodata] = np.nan

    # --------------------------------------------------------
    # Soporte SUHI definitivo
    # --------------------------------------------------------

    finite_common = (
        np.isfinite(obs)
        & np.isfinite(pred)
    )

    suhi_common = (
        fixed_test_support
        & finite_common
    )

    assert int(suhi_common.sum()) == EXPECTED_VALID[year]


    # --------------------------------------------------------
    # Referencia anual
    # --------------------------------------------------------

    ref_valid = (
        R_disk
        & suhi_common
    )

    tref_obs = float(
        np.median(obs[ref_valid])
    )

    tref_pred = float(
        np.median(pred[ref_valid])
    )


    # --------------------------------------------------------
    # SUHI
    # --------------------------------------------------------

    suhi_obs = (
        obs - tref_obs
    )

    suhi_pred = (
        pred - tref_pred
    )

    suhi_residual = (
        suhi_pred - suhi_obs
    )


    # --------------------------------------------------------
    # Aplicar exactamente la misma máscara a los 3 productos
    # --------------------------------------------------------

    suhi_obs_out = np.full(
        obs.shape,
        NODATA_OUT,
        dtype="float32"
    )

    suhi_pred_out = np.full(
        obs.shape,
        NODATA_OUT,
        dtype="float32"
    )

    suhi_residual_out = np.full(
        obs.shape,
        NODATA_OUT,
        dtype="float32"
    )

    suhi_obs_out[suhi_common] = (
        suhi_obs[suhi_common]
        .astype("float32")
    )

    suhi_pred_out[suhi_common] = (
        suhi_pred[suhi_common]
        .astype("float32")
    )

    suhi_residual_out[suhi_common] = (
        suhi_residual[suhi_common]
        .astype("float32")
    )


    # --------------------------------------------------------
    # Perfil de salida
    # --------------------------------------------------------

    profile.update(
        dtype="float32",
        count=1,
        nodata=NODATA_OUT,
        compress="deflate"
    )


    # --------------------------------------------------------
    # Nombres definitivos
    # --------------------------------------------------------

    out_obs = (
        OUTPUT_DIR
        / f"SUHI_observed_C_{year}.tif"
    )

    out_pred = (
        OUTPUT_DIR
        / f"SUHI_predicted_C_{year}.tif"
    )

    out_res = (
        OUTPUT_DIR
        / f"SUHI_residual_pred_minus_obs_C_{year}.tif"
    )


    # --------------------------------------------------------
    # Escritura
    # --------------------------------------------------------

    with rasterio.open(
        out_obs,
        "w",
        **profile
    ) as dst:
        dst.write(
            suhi_obs_out,
            1
        )

    with rasterio.open(
        out_pred,
        "w",
        **profile
    ) as dst:
        dst.write(
            suhi_pred_out,
            1
        )

    with rasterio.open(
        out_res,
        "w",
        **profile
    ) as dst:
        dst.write(
            suhi_residual_out,
            1
        )


    export_rows.append({
        "year": year,
        "n_valid": int(suhi_common.sum()),
        "n_reference": int(ref_valid.sum()),
        "tref_obs_c": tref_obs,
        "tref_pred_c": tref_pred,
        "observed_file": out_obs.name,
        "predicted_file": out_pred.name,
        "residual_file": out_res.name,
    })


# ------------------------------------------------------------
# 3. Resumen exportación
# ------------------------------------------------------------

suhi_export_summary = pd.DataFrame(
    export_rows
)

print("EXPORTACIÓN COMPLETADA")
print("======================")

display(
    suhi_export_summary.round(4)
)

In [ ]:
# ============================================================
# FASE F — F3. Auditoría física de los 9 GeoTIFF SUHI
# ============================================================

audit_rows = []

EXPECTED_SHAPE = (1115, 1257)

for year in YEARS:

    paths = {
        "observed":
            OUTPUT_DIR
            / f"SUHI_observed_C_{year}.tif",

        "predicted":
            OUTPUT_DIR
            / f"SUHI_predicted_C_{year}.tif",

        "residual":
            OUTPUT_DIR
            / f"SUHI_residual_pred_minus_obs_C_{year}.tif",
    }

    valid_masks = {}

    for product, path in paths.items():

        with rasterio.open(path) as src:

            arr = src.read(1)

            valid = (
                np.isfinite(arr)
                & (arr != src.nodata)
            )

            valid_masks[product] = valid

            vals = arr[valid]

            audit_rows.append({
                "year": year,
                "product": product,
                "crs": str(src.crs),
                "height": src.height,
                "width": src.width,
                "res_x": src.res[0],
                "res_y": src.res[1],
                "dtype": src.dtypes[0],
                "nodata": src.nodata,
                "n_valid": int(valid.sum()),
                "min_c": float(np.min(vals)),
                "median_c": float(np.median(vals)),
                "max_c": float(np.max(vals)),
            })

            assert src.crs.to_epsg() == 32618
            assert src.shape == EXPECTED_SHAPE
            assert src.res == (30.0, 30.0)
            assert src.dtypes[0] == "float32"
            assert src.nodata == -9999.0

    # Los tres productos deben compartir exactamente la máscara
    assert np.array_equal(
        valid_masks["observed"],
        valid_masks["predicted"]
    )

    assert np.array_equal(
        valid_masks["observed"],
        valid_masks["residual"]
    )

    assert int(
        valid_masks["observed"].sum()
    ) == EXPECTED_VALID[year]


suhi_raster_audit = pd.DataFrame(
    audit_rows
)

display(
    suhi_raster_audit.round(4)
)

print(
    "\nAUDITORÍA DE LOS 9 GEOTIFF SUHI: OK"
)

In [ ]:
# ============================================================
# FASE F — F4. Construir suhi_summary_final.csv
# ============================================================

import numpy as np
import pandas as pd
from pathlib import Path

OUTPUT_DIR = Path(
    "/content/drive/MyDrive/"
    "Barranquilla_LST_2013_2026_Modeling/03_outputs"
)

# ------------------------------------------------------------
# 1. Configuración nominal A_min = 0.333 km²
# ------------------------------------------------------------

nominal = sensitivity_suhi[
    np.isclose(
        sensitivity_suhi["A_min_km2"],
        0.333
    )
].copy()

# ------------------------------------------------------------
# 2. Construir tabla final
# ------------------------------------------------------------

suhi_summary_final = nominal[
    [
        "year",
        "n_R_valid",
        "n_U_valid",
        "tref_obs_c",
        "tref_pred_c",
        "tref_error_c",
        "urban_median_obs_c",
        "urban_median_pred_c",
        "deltaT_obs_c",
        "deltaT_pred_c",
        "deltaT_error_c",
        "p95_urban_obs_c",
        "p95_urban_pred_c",
        "p95_urban_error_c",
    ]
].copy()

# Incorporar soporte SUHI full-domain definitivo
support_counts = support_audit[
    [
        "year",
        "n_suhi_common",
    ]
].copy()

suhi_summary_final = (
    suhi_summary_final
    .merge(
        support_counts,
        on="year",
        how="left"
    )
)

# Reordenar
suhi_summary_final = suhi_summary_final[
    [
        "year",
        "n_suhi_common",
        "n_R_valid",
        "n_U_valid",
        "tref_obs_c",
        "tref_pred_c",
        "tref_error_c",
        "urban_median_obs_c",
        "urban_median_pred_c",
        "deltaT_obs_c",
        "deltaT_pred_c",
        "deltaT_error_c",
        "p95_urban_obs_c",
        "p95_urban_pred_c",
        "p95_urban_error_c",
    ]
]

display(
    suhi_summary_final.round(4)
)

# ------------------------------------------------------------
# 3. Guardar
# ------------------------------------------------------------

summary_path = (
    OUTPUT_DIR
    / "suhi_summary_final.csv"
)

suhi_summary_final.to_csv(
    summary_path,
    index=False
)

print("\nGuardado:")
print(summary_path)

In [ ]:
# ============================================================
# FASE F — F5. Guardar parámetros finales SUHI
# ============================================================

import json

parameters = {

    "emc_built_epoch": 2022,
    "emc_built_release": "R2025A",

    "crs": "EPSG:32618",
    "pixel_size_m": 30,

    "built_fraction_threshold": 0.10,

    "connectivity": 8,

    "core_min_area_km2": 0.333,

    "reference_band_km": [4.0, 6.0],

    "test_years": [
        2023,
        2024,
        2025
    ],

    "reference_pixels_geometry": 167527,
    "reference_area_km2": 150.7743,

    "urban_support_definition":
        "BU_core intersect annual SUHI common support",

    "reference_definition":
        "4 < distance_to_BU_core <= 6 km "
        "intersect fixed TEST support "
        "excluding BU_all",

    "suhi_support_definition":
        "fixed TEST support intersect finite observed/predicted LST",

    "p95_definition":
        "95th percentile of SUHI within BU_core",

    "suhi_residual_definition":
        "SUHI_predicted - SUHI_observed",

    "suhi_valid_pixels": {
        "2023": 904942,
        "2024": 905134,
        "2025": 905276
    }
}

json_path = (
    OUTPUT_DIR
    / "suhi_reference_parameters.json"
)

with open(
    json_path,
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        parameters,
        f,
        indent=2,
        ensure_ascii=False
    )

print("Guardado:")
print(json_path)

In [ ]:
# ============================================================
# FASE F — F6. Auditoría final CSV + JSON
# ============================================================

import json
import pandas as pd

# CSV
summary_check = pd.read_csv(
    OUTPUT_DIR
    / "suhi_summary_final.csv"
)

print("SUHI SUMMARY FINAL")
print("==================")
display(
    summary_check.round(4)
)

# JSON
with open(
    OUTPUT_DIR
    / "suhi_reference_parameters.json",
    "r",
    encoding="utf-8"
) as f:
    params_check = json.load(f)

print("\nPARAMETERS")
print("==========")
print(
    json.dumps(
        params_check,
        indent=2,
        ensure_ascii=False
    )
)

# Controles críticos
assert list(summary_check["year"]) == [2023, 2024, 2025]

assert params_check["core_min_area_km2"] == 0.333
assert params_check["reference_pixels_geometry"] == 167527
assert params_check["p95_definition"] == (
    "95th percentile of SUHI within BU_core"
)

assert params_check["suhi_valid_pixels"]["2023"] == 904942
assert params_check["suhi_valid_pixels"]["2024"] == 905134
assert params_check["suhi_valid_pixels"]["2025"] == 905276

print(
    "\nAUDITORÍA CSV + JSON: OK"
)

## 15. Fase G — Figura editorial SUHI 2025


In [ ]:
# ============================================================
# FASE G — G2. FIGURA EDITORIAL SUHI 2025
# BLOQUE COMPLETO Y AUTÓNOMO — VERSIÓN FINAL
#
# Formato coherente con Figure 5:
#   (a) Observed SUHI
#   (b) Predicted SUHI
#   (c) Residual predicted - observed
#   (d) Urban SUHI intensity distribution — KDE
#
# Paneles (a), (b) y (c): CONGELADOS
#
# Panel (d):
#   - KDE formal con scipy.stats.gaussian_kde
#   - Scott's rule para bandwidth
#   - todos los píxeles urbanos válidos
#   - medianas y P95 observados/predichos
#   - jerarquía gráfica refinada:
#       KDE    = 2.0
#       Median = 1.3, alpha 0.90
#       P95    = 1.5, alpha 0.90
# ============================================================

from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import rasterio

from scipy import ndimage
from scipy.stats import gaussian_kde

from matplotlib.colors import TwoSlopeNorm
from matplotlib.ticker import FormatStrFormatter
from matplotlib.lines import Line2D


# ============================================================
# 1. CONFIGURACIÓN GENERAL
# ============================================================

BASE = Path(
    "/content/drive/MyDrive/"
    "Barranquilla_LST_2013_2026_Modeling"
)

OUTPUT_DIR = BASE / "03_outputs"

FIG_DIR = OUTPUT_DIR / "figures"
FIG_DIR.mkdir(
    parents=True,
    exist_ok=True
)

YEAR = 2025

EXPORT_DPI = 600

SUHI_CMAP = "turbo"
RESIDUAL_CMAP = "RdBu_r"


# ============================================================
# 2. RUTAS
# ============================================================

OBS_PATH = (
    OUTPUT_DIR
    / f"SUHI_observed_C_{YEAR}.tif"
)

PRED_PATH = (
    OUTPUT_DIR
    / f"SUHI_predicted_C_{YEAR}.tif"
)

RES_PATH = (
    OUTPUT_DIR
    / f"SUHI_residual_pred_minus_obs_C_{YEAR}.tif"
)

FRACTION_PATH = (
    BASE
    / "01_aoi_reference"
    / "emc_built_2022"
    / "processed"
    / "EMC_BUILT_fraction_E2022_EPSG32618_30m.tif"
)


# ============================================================
# 3. FUNCIONES
# ============================================================

def read_raster(path):
    """
    Lee un raster de una banda, convierte NoData a NaN
    y devuelve array + metadatos básicos.
    """

    if not path.exists():
        raise FileNotFoundError(
            f"No existe el archivo:\n{path}"
        )

    with rasterio.open(path) as src:

        arr = src.read(1).astype("float64")

        nodata = src.nodata

        if nodata is not None:
            arr[arr == nodata] = np.nan

        meta = {
            "crs": src.crs,
            "shape": src.shape,
            "transform": src.transform,
            "res": src.res,
            "bounds": src.bounds,
        }

    return arr, meta


def draw_north_arrow(
    ax,
    x_frac=0.085,
    y_frac=0.89,
    length_frac=0.15
):
    """
    Flecha norte similar al estilo usado en Figure 5.
    """

    xmin, xmax = ax.get_xlim()
    ymin, ymax = ax.get_ylim()

    x = xmin + x_frac * (xmax - xmin)

    y_top = (
        ymin
        + y_frac * (ymax - ymin)
    )

    y_bottom = (
        y_top
        - length_frac * (ymax - ymin)
    )

    ax.annotate(
        "",
        xy=(x, y_top),
        xytext=(x, y_bottom),
        arrowprops=dict(
            facecolor="black",
            edgecolor="black",
            width=2.0,
            headwidth=9,
            headlength=11,
        ),
        zorder=20
    )

    ax.text(
        x,
        y_top + 0.025 * (ymax - ymin),
        "N",
        ha="center",
        va="bottom",
        fontsize=10,
        fontweight="bold"
    )


def draw_scalebar(
    ax,
    length_km=10,
    x_frac=0.08,
    y_frac=0.065
):
    """
    Barra de escala estilo Figure 5:
    - línea negra
    - halo blanco estrecho
    - marcas terminales
    - fondo blanco sutil detrás del texto
    """

    import matplotlib.patheffects as pe

    xmin, xmax = ax.get_xlim()
    ymin, ymax = ax.get_ylim()

    x_start = (
        xmin
        + x_frac * (xmax - xmin)
    )

    x_end = (
        x_start
        + length_km
    )

    y = (
        ymin
        + y_frac * (ymax - ymin)
    )

    tick_height = (
        0.014 * (ymax - ymin)
    )

    # --------------------------------------------------------
    # Halo blanco
    # --------------------------------------------------------

    halo = [
        pe.Stroke(
            linewidth=7.0,
            foreground="white"
        ),
        pe.Normal()
    ]

    # --------------------------------------------------------
    # Línea principal
    # --------------------------------------------------------

    line, = ax.plot(
        [x_start, x_end],
        [y, y],
        color="black",
        linewidth=3.2,
        solid_capstyle="butt",
        zorder=20
    )

    line.set_path_effects(
        halo
    )

    # --------------------------------------------------------
    # Marcas terminales
    # --------------------------------------------------------

    left_tick, = ax.plot(
        [x_start, x_start],
        [
            y - tick_height,
            y + tick_height
        ],
        color="black",
        linewidth=1.8,
        zorder=21
    )

    right_tick, = ax.plot(
        [x_end, x_end],
        [
            y - tick_height,
            y + tick_height
        ],
        color="black",
        linewidth=1.8,
        zorder=21
    )

    tick_halo = [
        pe.Stroke(
            linewidth=5.0,
            foreground="white"
        ),
        pe.Normal()
    ]

    left_tick.set_path_effects(
        tick_halo
    )

    right_tick.set_path_effects(
        tick_halo
    )

    # --------------------------------------------------------
    # Texto
    # --------------------------------------------------------

    ax.text(
        (x_start + x_end) / 2,
        y + 0.020 * (ymax - ymin),
        f"{length_km} km",
        ha="center",
        va="bottom",
        fontsize=9,
        color="black",
        zorder=22,
        bbox=dict(
            boxstyle="square,pad=0.10",
            facecolor="white",
            edgecolor="none",
            alpha=0.88
        )
    )


# ============================================================
# 4. LEER SUHI DEFINITIVO
# ============================================================

suhi_obs, meta_obs = read_raster(
    OBS_PATH
)

suhi_pred, meta_pred = read_raster(
    PRED_PATH
)

suhi_resid, meta_res = read_raster(
    RES_PATH
)


# ------------------------------------------------------------
# Auditoría geométrica
# ------------------------------------------------------------

assert (
    meta_obs["shape"]
    == meta_pred["shape"]
    == meta_res["shape"]
)

assert (
    meta_obs["transform"]
    == meta_pred["transform"]
    == meta_res["transform"]
)

assert (
    meta_obs["crs"]
    == meta_pred["crs"]
    == meta_res["crs"]
)

assert (
    meta_obs["crs"].to_epsg()
    == 32618
)


# ============================================================
# 5. SOPORTE COMÚN
# ============================================================

common_mask = (
    np.isfinite(suhi_obs)
    & np.isfinite(suhi_pred)
    & np.isfinite(suhi_resid)
)

n_common = int(
    common_mask.sum()
)

print(
    "Píxeles comunes:",
    n_common
)

assert (
    n_common
    == 905276
)


# ============================================================
# 6. EXTENSIÓN EN KM
# ============================================================

bounds = meta_obs["bounds"]

extent_km = [
    bounds.left / 1000.0,
    bounds.right / 1000.0,
    bounds.bottom / 1000.0,
    bounds.top / 1000.0,
]


# ============================================================
# 7. RECONSTRUIR BU_core
# ============================================================

fraction, meta_fraction = read_raster(
    FRACTION_PATH
)

assert (
    meta_fraction["shape"]
    == meta_obs["shape"]
)

assert (
    meta_fraction["transform"]
    == meta_obs["transform"]
)

assert (
    meta_fraction["crs"]
    == meta_obs["crs"]
)


BU_THRESHOLD = 0.10
CORE_MIN_AREA_KM2 = 0.333
PIXEL_SIZE_M = 30.0

PIXEL_AREA_KM2 = (
    PIXEL_SIZE_M
    * PIXEL_SIZE_M
    / 1e6
)


# ------------------------------------------------------------
# BU_all
# ------------------------------------------------------------

BU_all = (
    np.isfinite(fraction)
    & (fraction >= BU_THRESHOLD)
)


# ------------------------------------------------------------
# Componentes conectados — 8 vecinos
# ------------------------------------------------------------

structure = np.ones(
    (3, 3),
    dtype=int
)

labels, n_components_total = (
    ndimage.label(
        BU_all,
        structure=structure
    )
)

pixel_counts = np.bincount(
    labels.ravel()
)

component_areas_km2 = (
    pixel_counts
    * PIXEL_AREA_KM2
)

keep_labels = np.where(
    component_areas_km2
    >= CORE_MIN_AREA_KM2
)[0]

# quitar background
keep_labels = (
    keep_labels[
        keep_labels != 0
    ]
)

BU_core = np.isin(
    labels,
    keep_labels
)

print(
    "Componentes BU_core:",
    len(keep_labels)
)

assert (
    len(keep_labels)
    == 18
)


# ============================================================
# 8. SOPORTE URBANO FINAL
# ============================================================

urban_mask = (
    BU_core
    & common_mask
)

n_urban = int(
    urban_mask.sum()
)

print(
    "Píxeles urbanos:",
    n_urban
)

assert (
    n_urban
    == 113436
)


urban_obs = (
    suhi_obs[
        urban_mask
    ]
)

urban_pred = (
    suhi_pred[
        urban_mask
    ]
)


# ============================================================
# 9. ESTADÍSTICOS URBANOS
# ============================================================

median_obs = float(
    np.median(
        urban_obs
    )
)

median_pred = float(
    np.median(
        urban_pred
    )
)

p95_obs = float(
    np.percentile(
        urban_obs,
        95
    )
)

p95_pred = float(
    np.percentile(
        urban_pred,
        95
    )
)


print(
    "\nEstadísticos urbanos 2025"
)

print(
    "-------------------------"
)

print(
    "Observed median :",
    round(median_obs, 4)
)

print(
    "Predicted median:",
    round(median_pred, 4)
)

print(
    "Observed P95    :",
    round(p95_obs, 4)
)

print(
    "Predicted P95   :",
    round(p95_pred, 4)
)


# ------------------------------------------------------------
# Controles contra valores congelados
# ------------------------------------------------------------

assert np.isclose(
    median_obs,
    8.7313,
    atol=0.001
)

assert np.isclose(
    median_pred,
    7.3027,
    atol=0.001
)

assert np.isclose(
    p95_obs,
    11.6230,
    atol=0.001
)

assert np.isclose(
    p95_pred,
    8.7999,
    atol=0.001
)


# ============================================================
# 10. ESCALA SUHI OBSERVADO/PREDICHO
#
# P2–P98 del conjunto observado + predicho
# ============================================================

combined_suhi = np.concatenate(
    [
        suhi_obs[
            common_mask
        ],
        suhi_pred[
            common_mask
        ]
    ]
)

vmin_suhi = float(
    np.percentile(
        combined_suhi,
        2
    )
)

vmax_suhi = float(
    np.percentile(
        combined_suhi,
        98
    )
)


# ------------------------------------------------------------
# Garantizar que 0 esté visible
# ------------------------------------------------------------

vmin_suhi = min(
    vmin_suhi,
    0.0
)

vmax_suhi = max(
    vmax_suhi,
    0.0
)


print(
    "\nEscala SUHI obs/pred:"
)

print(
    "vmin =",
    round(vmin_suhi, 3)
)

print(
    "vmax =",
    round(vmax_suhi, 3)
)


# ------------------------------------------------------------
# Ticks manuales limpios
# ------------------------------------------------------------

suhi_ticks = [
    -3,
    0,
    2,
    4,
    6,
    8,
    10
]

suhi_ticks = [
    t
    for t in suhi_ticks
    if vmin_suhi <= t <= vmax_suhi
]


# ============================================================
# 11. ESCALA RESIDUAL
#
# ± P98(|residual|)
# ============================================================

residual_values = (
    suhi_resid[
        common_mask
    ]
)

res_lim = float(
    np.percentile(
        np.abs(
            residual_values
        ),
        98
    )
)

res_norm = TwoSlopeNorm(
    vmin=-res_lim,
    vcenter=0.0,
    vmax=res_lim
)


print(
    "\nEscala residual:"
)

print(
    "±",
    round(res_lim, 3),
    "°C"
)


# ============================================================
# 12. DISTRIBUCIÓN URBANA — KDE FORMAL
# ============================================================

# ------------------------------------------------------------
# Rango gráfico común
#
# La KDE usa TODOS los píxeles urbanos válidos.
#
# P0.5–P99.5 se utiliza únicamente como rango gráfico.
# No se eliminan esos valores para calcular la KDE.
# ------------------------------------------------------------

urban_combined = np.concatenate(
    [
        urban_obs,
        urban_pred
    ]
)

dist_min = float(
    np.percentile(
        urban_combined,
        0.5
    )
)

dist_max = float(
    np.percentile(
        urban_combined,
        99.5
    )
)

x_kde = np.linspace(
    dist_min,
    dist_max,
    500
)


# ------------------------------------------------------------
# KDE observada
# ------------------------------------------------------------

kde_obs = gaussian_kde(
    urban_obs,
    bw_method="scott"
)

density_obs = kde_obs(
    x_kde
)


# ------------------------------------------------------------
# KDE predicha
# ------------------------------------------------------------

kde_pred = gaussian_kde(
    urban_pred,
    bw_method="scott"
)

density_pred = kde_pred(
    x_kde
)


# ------------------------------------------------------------
# Control KDE
# ------------------------------------------------------------

assert np.all(
    np.isfinite(
        density_obs
    )
)

assert np.all(
    np.isfinite(
        density_pred
    )
)


print(
    "\nKDE urbana 2025"
)

print(
    "----------------"
)

print(
    "Método: scipy.stats.gaussian_kde"
)

print(
    "Bandwidth: Scott's rule"
)

print(
    "Observed n:",
    len(urban_obs)
)

print(
    "Predicted n:",
    len(urban_pred)
)

print(
    "Display range:",
    round(dist_min, 3),
    "to",
    round(dist_max, 3),
    "°C"
)


# ============================================================
# 13. ESTILO GENERAL
# ============================================================

plt.rcParams.update({
    "font.family": "DejaVu Sans",
    "font.size": 10,
    "axes.titlesize": 11,
    "axes.labelsize": 9.5,
    "xtick.labelsize": 8.5,
    "ytick.labelsize": 8.5,
    "legend.fontsize": 8.5,
    "figure.dpi": 120,
    "savefig.dpi": EXPORT_DPI,
})


# ============================================================
# 14. CREAR FIGURA
# ============================================================

fig, axes = plt.subplots(
    2,
    2,
    figsize=(12.7, 10.2)
)

fig.subplots_adjust(
    left=0.07,
    right=0.92,
    bottom=0.07,
    top=0.96,
    wspace=0.26,
    hspace=0.24
)


# ============================================================
# PANEL (a) — OBSERVED SUHI
# CONGELADO
# ============================================================

im_a = axes[0, 0].imshow(
    suhi_obs,
    extent=extent_km,
    origin="upper",
    cmap=SUHI_CMAP,
    vmin=vmin_suhi,
    vmax=vmax_suhi,
)

axes[0, 0].set_title(
    "(a) Observed SUHI",
    fontweight="bold"
)

axes[0, 0].set_xlabel(
    "Easting (km, UTM 18N)"
)

axes[0, 0].set_ylabel(
    "Northing (km, UTM 18N)"
)

axes[0, 0].set_aspect(
    "equal"
)

axes[0, 0].grid(
    alpha=0.10
)

draw_north_arrow(
    axes[0, 0]
)

draw_scalebar(
    axes[0, 0],
    length_km=10
)


# ============================================================
# PANEL (b) — PREDICTED SUHI
# CONGELADO
# ============================================================

im_b = axes[0, 1].imshow(
    suhi_pred,
    extent=extent_km,
    origin="upper",
    cmap=SUHI_CMAP,
    vmin=vmin_suhi,
    vmax=vmax_suhi,
)

axes[0, 1].set_title(
    "(b) Predicted SUHI",
    fontweight="bold"
)

axes[0, 1].set_xlabel(
    "Easting (km, UTM 18N)"
)

axes[0, 1].set_ylabel(
    ""
)

axes[0, 1].set_aspect(
    "equal"
)

axes[0, 1].grid(
    alpha=0.10
)


# ------------------------------------------------------------
# Colorbar común SUHI
# ------------------------------------------------------------

cbar_suhi = fig.colorbar(
    im_b,
    ax=[
        axes[0, 0],
        axes[0, 1]
    ],
    orientation="vertical",
    fraction=0.025,
    pad=0.015
)

cbar_suhi.set_label(
    "SUHI (°C)"
)

cbar_suhi.set_ticks(
    suhi_ticks
)

cbar_suhi.ax.yaxis.set_major_formatter(
    FormatStrFormatter(
        "%.1f"
    )
)


# ============================================================
# PANEL (c) — RESIDUAL
# CONGELADO
# ============================================================

im_c = axes[1, 0].imshow(
    suhi_resid,
    extent=extent_km,
    origin="upper",
    cmap=RESIDUAL_CMAP,
    norm=res_norm
)

axes[1, 0].set_title(
    "(c) Residual (predicted − observed)",
    fontweight="bold"
)

axes[1, 0].set_xlabel(
    "Easting (km, UTM 18N)"
)

axes[1, 0].set_ylabel(
    "Northing (km, UTM 18N)"
)

axes[1, 0].set_aspect(
    "equal"
)

axes[1, 0].grid(
    alpha=0.10
)


# ------------------------------------------------------------
# Colorbar residual
# ------------------------------------------------------------

cbar_res = fig.colorbar(
    im_c,
    ax=axes[1, 0],
    orientation="vertical",
    fraction=0.046,
    pad=0.02
)

cbar_res.set_label(
    "Residual (°C)"
)

cbar_res.ax.yaxis.set_major_formatter(
    FormatStrFormatter(
        "%.1f"
    )
)


# ============================================================
# PANEL (d) — DISTRIBUCIÓN URBANA
# KDE FORMAL
# ============================================================

ax = axes[1, 1]


# ------------------------------------------------------------
# Observed KDE
# ------------------------------------------------------------

ax.fill_between(
    x_kde,
    0,
    density_obs,
    color="tab:blue",
    alpha=0.25,
    linewidth=0
)

ax.plot(
    x_kde,
    density_obs,
    color="tab:blue",
    linewidth=2.0,
    label="Observed"
)


# ------------------------------------------------------------
# Predicted KDE
# ------------------------------------------------------------

ax.fill_between(
    x_kde,
    0,
    density_pred,
    color="tab:orange",
    alpha=0.25,
    linewidth=0
)

ax.plot(
    x_kde,
    density_pred,
    color="tab:orange",
    linewidth=2.0,
    label="Predicted"
)


# ------------------------------------------------------------
# Medianas
# Línea discontinua
# ------------------------------------------------------------

ax.axvline(
    median_obs,
    color="tab:blue",
    linestyle="--",
    linewidth=1.3,
    alpha=0.90,
    zorder=5
)

ax.axvline(
    median_pred,
    color="tab:orange",
    linestyle="--",
    linewidth=1.3,
    alpha=0.90,
    zorder=5
)


# ------------------------------------------------------------
# P95
# Línea punteada
# ------------------------------------------------------------

ax.axvline(
    p95_obs,
    color="tab:blue",
    linestyle=":",
    linewidth=1.5,
    alpha=0.90,
    zorder=5
)

ax.axvline(
    p95_pred,
    color="tab:orange",
    linestyle=":",
    linewidth=1.5,
    alpha=0.90,
    zorder=5
)


# ------------------------------------------------------------
# Título y ejes
# ------------------------------------------------------------

ax.set_title(
    "(d) Urban SUHI intensity distribution",
    fontweight="bold"
)

ax.set_xlabel(
    "SUHI (°C)"
)

ax.set_ylabel(
    "Density"
)

ax.grid(
    alpha=0.15
)

ax.set_xlim(
    dist_min,
    dist_max
)


# ------------------------------------------------------------
# Caja estadística
# ------------------------------------------------------------

stats_text = (
    f"Observed median = {median_obs:.2f} °C\n"
    f"Predicted median = {median_pred:.2f} °C\n"
    f"Observed P95 = {p95_obs:.2f} °C\n"
    f"Predicted P95 = {p95_pred:.2f} °C\n"
    f"n = {n_urban:,} pixels"
)

ax.text(
    0.03,
    0.97,
    stats_text,
    transform=ax.transAxes,
    va="top",
    ha="left",
    fontsize=8.8,
    bbox=dict(
        boxstyle="round,pad=0.35",
        facecolor="white",
        edgecolor="0.6",
        alpha=0.92
    )
)


# ------------------------------------------------------------
# Leyenda
#
# Color:
#   azul    = observed
#   naranja = predicted
#
# Estilo:
#   -- = median
#   :  = P95
# ------------------------------------------------------------

legend_handles = [

    Line2D(
        [0],
        [0],
        color="tab:blue",
        linewidth=2.0,
        label="Observed"
    ),

    Line2D(
        [0],
        [0],
        color="tab:orange",
        linewidth=2.0,
        label="Predicted"
    ),

    Line2D(
        [0],
        [0],
        color="black",
        linestyle="--",
        linewidth=1.3,
        alpha=0.90,
        label="Median"
    ),

    Line2D(
        [0],
        [0],
        color="black",
        linestyle=":",
        linewidth=1.5,
        alpha=0.90,
        label="P95"
    )
]

ax.legend(
    handles=legend_handles,
    loc="upper right",
    frameon=True
)


# ============================================================
# 15. GUARDAR FIGURA
# ============================================================

PNG_PATH = (
    FIG_DIR
    / "Figure_SUHI_2025_FINAL.png"
)

PDF_PATH = (
    FIG_DIR
    / "Figure_SUHI_2025_FINAL.pdf"
)


fig.savefig(
    PNG_PATH,
    dpi=EXPORT_DPI,
    bbox_inches="tight"
)

fig.savefig(
    PDF_PATH,
    dpi=EXPORT_DPI,
    bbox_inches="tight"
)

plt.show()


# ============================================================
# 16. RESUMEN
# ============================================================

print(
    "\nFIGURA GENERADA"
)

print(
    "================"
)

print(
    "PNG:",
    PNG_PATH
)

print(
    "PDF:",
    PDF_PATH
)


print(
    "\nControles:"
)

print(
    "Common SUHI pixels:",
    n_common
)

print(
    "Urban pixels:",
    n_urban
)

print(
    "Observed median:",
    round(
        median_obs,
        4
    )
)

print(
    "Predicted median:",
    round(
        median_pred,
        4
    )
)

print(
    "Observed P95:",
    round(
        p95_obs,
        4
    )
)

print(
    "Predicted P95:",
    round(
        p95_pred,
        4
    )
)

print(
    "KDE bandwidth:",
    "Scott's rule"
)